## Clasificación multiclase de consultas bancarias con DistilRoBERTa e interpretación de errores mediante Falcon-7B-Instruct

---

## Objetivo

Implementar y evaluar un modelo Transformer basado en DistilRoBERTa
para la clasificación multiclase de consultas bancarias del dataset
Banking77. Posteriormente, analizar los errores de clasificación y
utilizar un modelo de lenguaje grande (Falcon-7B-Instruct) para generar
explicaciones interpretables de las predicciones incorrectas.

El trabajo se estructura en los siguientes apartados:

0. Configuración del entorno
1. Análisis exploratorio de datos (EDA).
2. Fine-tuning y evaluación de DistilRoBERTa.
3. Diseño y calibración de un prompt para Falcon-7B-Instruct.
4. Conclusiones
5. Referencias

# 0. Configuración del entorno

En esta sección se configuran las dependencias y los recursos computacionales
necesarios para ejecutar el experimento completo. El notebook fue diseñado
para ejecutarse de principio a fin en Google Colab utilizando una GPU NVIDIA
Tesla T4.

El mismo entorno de ejecución se utilizará para el análisis exploratorio de
datos, el fine-tuning de DistilRoBERTa y la inferencia con
Falcon-7B-Instruct, favoreciendo la reproducibilidad del flujo experimental.

In [ ]:
# ============================================================
# 0.1 INSTALACIÓN DE DEPENDENCIAS
# ============================================================

!pip -q install \
    "datasets==3.6.0" \
    transformers \
    accelerate \
    evaluate \
    bitsandbytes \
    wordcloud

In [ ]:
# ============================================================
# 0.2 VERIFICACIÓN DEL ENTORNO
# ============================================================

import torch
import datasets
import transformers

print("PyTorch:", torch.__version__)
print("Datasets:", datasets.__version__)
print("Transformers:", transformers.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM total:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

In [ ]:
# ============================================================
# 0.3 IMPORTACIÓN DE LIBRERÍAS Y REPRODUCIBILIDAD
# ============================================================

# Librerías generales
import re
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# Procesamiento y visualización de texto
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from wordcloud import WordCloud

# Procesamiento de lenguaje natural
import nltk
from nltk.corpus import stopwords

# Hugging Face
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

# Reproducibilidad
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Semilla global configurada: {SEED}")

# 1. Análisis Exploratorio de Datos (EDA)

En esta sección se analiza el conjunto de datos **Banking77**, compuesto por
consultas bancarias breves en inglés clasificadas en 77 intenciones.

El análisis exploratorio permitirá conocer las características principales de
los textos, su longitud, vocabulario y distribución entre clases. Además, se
buscarán posibles características del conjunto de datos que posteriormente
puedan ayudar a interpretar el comportamiento del modelo de clasificación.

## 1.1 Carga e inspección inicial del dataset

Se carga el conjunto de datos `PolyAI/banking77` desde Hugging Face y se
inspeccionan sus particiones, estructura, número de observaciones y categorías.

Banking77 es un conjunto de datos de detección de intenciones en el dominio
bancario compuesto por 13,083 consultas distribuidas en 77 categorías
(Casanueva et al., 2020).

In [ ]:
# ============================================================
# CARGA DEL DATASET BANKING77
# ============================================================

dataset = load_dataset(
    "PolyAI/banking77",
    trust_remote_code=True
)

print(dataset)

In [ ]:
# ============================================================
# INSPECCIÓN INICIAL DEL DATASET
# ============================================================

print("Primeras 5 muestras del conjunto de entrenamiento:\n")

for i in range(5):
    print(f"Consulta {i + 1}: {dataset['train'][i]['text']}")
    print(f"Label: {dataset['train'][i]['label']}")
    print("-" * 60)

In [ ]:
# ============================================================
# CLASES / INTENCIONES
# ============================================================

label_names = dataset["train"].features["label"].names

print(f"Número total de clases: {len(label_names)}")
print("\nPrimeras 15 clases:\n")

for i, label in enumerate(label_names[:15]):
    print(f"{i:2d} -> {label}")

In [ ]:
# ============================================================
# CONVERSIÓN A DATAFRAMES
# ============================================================

df_train = dataset["train"].to_pandas()
df_test = dataset["test"].to_pandas()

# Agregar el nombre de la intención correspondiente a cada label
df_train["intent"] = df_train["label"].apply(lambda x: label_names[x])
df_test["intent"] = df_test["label"].apply(lambda x: label_names[x])

print(f"Muestras de entrenamiento: {len(df_train):,}")
print(f"Muestras de prueba:        {len(df_test):,}")
print(f"Número de clases:          {len(label_names)}")

display(df_train.head())

In [ ]:
# ============================================================
# COMPROBACIÓN BÁSICA DE VALORES FALTANTES
# ============================================================

print("Valores faltantes:")
print(f"Train - texto: {df_train['text'].isna().sum()}")
print(f"Train - etiqueta: {df_train['label'].isna().sum()}")
print(f"Test  - texto: {df_test['text'].isna().sum()}")
print(f"Test  - etiqueta: {df_test['label'].isna().sum()}")

### Observaciones iniciales

El conjunto de datos contiene **13,083 consultas bancarias**, distribuidas en
10,003 muestras de entrenamiento y 3,080 muestras de prueba, clasificadas en
77 intenciones diferentes. No se identificaron valores faltantes en los textos
ni en las etiquetas de ninguna de las dos particiones, por lo que no es
necesario realizar un tratamiento adicional de datos ausentes antes del
análisis exploratorio.

## 1.2 Longitud y número de palabras de las consultas

Se analiza la extensión de las consultas utilizando dos métricas:

- **Longitud en caracteres:** número total de caracteres de cada consulta.
- **Número de palabras:** cantidad de palabras obtenidas al separar el texto
  por espacios.

El análisis se realiza tanto sobre entrenamiento como sobre prueba para
comparar las características de ambas particiones.

In [ ]:
# ============================================================
# LONGITUD Y NÚMERO DE PALABRAS
# ============================================================

# Longitud en caracteres
df_train["char_length"] = df_train["text"].str.len()
df_test["char_length"] = df_test["text"].str.len()

# Número de palabras
df_train["word_count"] = df_train["text"].str.split().str.len()
df_test["word_count"] = df_test["text"].str.split().str.len()

display(
    df_train[
        ["text", "intent", "char_length", "word_count"]
    ].head(10)
)

In [ ]:
# ============================================================
# ESTADÍSTICAS DESCRIPTIVAS
# ============================================================

print("ESTADÍSTICAS DEL CONJUNTO DE ENTRENAMIENTO")
display(
    df_train[["char_length", "word_count"]]
    .describe()
    .round(2)
)

print("\nESTADÍSTICAS DEL CONJUNTO DE PRUEBA")
display(
    df_test[["char_length", "word_count"]]
    .describe()
    .round(2)
)

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE LA LONGITUD EN CARACTERES
# ============================================================

plt.figure(figsize=(10, 5))

plt.hist(
    df_train["char_length"],
    bins=40,
    alpha=0.6,
    density=True,
    label="Train"
)

plt.hist(
    df_test["char_length"],
    bins=40,
    alpha=0.6,
    density=True,
    label="Test"
)

plt.axvline(
    df_train["char_length"].median(),
    linestyle="--",
    label=f"Mediana train = {df_train['char_length'].median():.0f}"
)

plt.axvline(
    df_test["char_length"].median(),
    linestyle=":",
    label=f"Mediana test = {df_test['char_length'].median():.0f}"
)

plt.title("Distribución de la longitud de las consultas")
plt.xlabel("Número de caracteres")
plt.ylabel("Densidad")
plt.legend()
plt.show()

In [ ]:
# ============================================================
# DISTRIBUCIÓN DEL NÚMERO DE PALABRAS
# ============================================================

plt.figure(figsize=(10, 5))

plt.hist(
    df_train["word_count"],
    bins=30,
    alpha=0.6,
    density=True,
    label="Train"
)

plt.hist(
    df_test["word_count"],
    bins=30,
    alpha=0.6,
    density=True,
    label="Test"
)

plt.axvline(
    df_train["word_count"].median(),
    linestyle="--",
    label=f"Mediana train = {df_train['word_count'].median():.0f}"
)

plt.axvline(
    df_test["word_count"].median(),
    linestyle=":",
    label=f"Mediana test = {df_test['word_count'].median():.0f}"
)

plt.title("Distribución del número de palabras por consulta")
plt.xlabel("Número de palabras")
plt.ylabel("Densidad")
plt.legend()
plt.show()

### Interpretación de la longitud de las consultas

Las consultas de Banking77 son predominantemente breves. En el conjunto de
entrenamiento presentan una media de **59.47 caracteres y 11.95 palabras**,
mientras que las medianas son de **47 caracteres y 10 palabras**. En el
conjunto de prueba se observa un comportamiento similar, con medias de
**54.23 caracteres y 10.95 palabras**, y medianas de **45 caracteres y
9 palabras**.

En ambas particiones las medias son superiores a las medianas y las
distribuciones presentan una cola hacia valores altos, lo que evidencia una
asimetría hacia la derecha producida por un número reducido de consultas
considerablemente más extensas. Los valores máximos alcanzan 433 caracteres
y 79 palabras en entrenamiento, y 368 caracteres y 69 palabras en prueba.

Las distribuciones normalizadas muestran además un comportamiento semejante
entre entrenamiento y prueba, por lo que no se observan diferencias importantes
en la longitud de las consultas entre ambas particiones.

Desde la perspectiva de la clasificación de intenciones, la brevedad de las
consultas puede representar un reto, ya que en muchos casos el modelo deberá
distinguir entre intenciones utilizando pocas palabras y un contexto limitado.
Esta observación será contrastada posteriormente con los errores obtenidos por
el modelo Transformer.

## 1.3 Limpieza básica del texto

Para realizar el análisis de frecuencias, n-gramas y la nube de palabras se
genera una versión normalizada de las consultas. El proceso incluye conversión
a minúsculas, eliminación de caracteres especiales y eliminación de stopwords
en inglés.

Esta transformación se utilizará exclusivamente para el análisis exploratorio.
El texto original se conservará para el posterior entrenamiento de
DistilRoBERTa, evitando eliminar información contextual que pueda resultar
relevante para la clasificación de intenciones.

In [ ]:
# ============================================================
# STOPWORDS EN INGLÉS
# ============================================================

nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))

# Las negaciones pueden modificar el significado de una consulta,
# por lo que se conservan durante la limpieza.
negations = {"no", "not", "nor"}
stop_words = stop_words - negations

print(f"Número de stopwords utilizadas: {len(stop_words)}")
print(list(stop_words)[:20])

In [ ]:
# ============================================================
# FUNCIÓN DE LIMPIEZA
# ============================================================

def clean_text(text):
    """
    Normaliza el texto para el análisis exploratorio:
    1. Convierte a minúsculas.
    2. Normaliza contracciones negativas.
    3. Elimina caracteres especiales y números.
    4. Elimina stopwords, preservando no/not/nor.
    """

    text = text.lower()

    # Casos irregulares de contracciones negativas
    text = re.sub(r"\bcan['’]t\b", "can not", text)
    text = re.sub(r"\bwon['’]t\b", "will not", text)

    # Contracciones negativas regulares:
    # isn't -> is not, hasn't -> has not, didn't -> did not...
    text = re.sub(r"n['’]t\b", " not", text)

    # Eliminar caracteres especiales y números
    text = re.sub(r"[^a-z\s]", " ", text)

    # Eliminar stopwords preservando las negaciones
    words = [
        word for word in text.split()
        if word not in stop_words
    ]

    return " ".join(words)

In [ ]:
# Aplicar limpieza sin modificar el texto original
df_train["clean_text"] = df_train["text"].apply(clean_text)
df_test["clean_text"] = df_test["text"].apply(clean_text)

# Comparación entre texto original y texto limpio
display(
    df_train[["text", "clean_text"]].head(15)
)

### Observaciones sobre la limpieza

La limpieza reduce las consultas a los términos con mayor contenido semántico,
facilitando el posterior análisis de frecuencias y n-gramas. Se conservaron
deliberadamente las negaciones `no`, `not` y `nor`, ya que su eliminación
podría modificar el significado de una consulta.

Antes de eliminar caracteres especiales se normalizaron las contracciones
negativas. Por ejemplo, `hasn't arrived` se transforma en `has not arrived`
y, después de eliminar las stopwords generales, se conserva como
`not arrived`. También se trataron explícitamente contracciones irregulares
como `can't` y `won't`, evitando generar tokens artificiales durante el
preprocesamiento.

La columna `clean_text` se utilizará únicamente para el análisis exploratorio.
Para el entrenamiento de DistilRoBERTa se conservará la columna `text`
original, permitiendo que el tokenizer del modelo procese la consulta completa
y mantenga su información contextual.

## 1.4 Análisis de frecuencia de palabras

A partir de las consultas normalizadas se identifican las 15 palabras con
mayor frecuencia en el conjunto de entrenamiento. Este análisis permite
reconocer el vocabulario predominante del dominio bancario y observar qué
términos aparecen de forma transversal en diferentes intenciones.

In [ ]:
# ============================================================
# 15 PALABRAS MÁS FRECUENTES
# ============================================================

# Unir todos los textos limpios del conjunto de entrenamiento
all_words = " ".join(df_train["clean_text"]).split()

# Contar la frecuencia de cada palabra
word_freq = Counter(all_words)

# Obtener las 15 palabras más frecuentes
top_15_words = word_freq.most_common(15)

# Convertir a DataFrame para facilitar su visualización
df_top_words = pd.DataFrame(
    top_15_words,
    columns=["word", "frequency"]
)

display(df_top_words)

In [ ]:
# ============================================================
# VISUALIZACIÓN DE LAS 15 PALABRAS MÁS FRECUENTES
# ============================================================

plt.figure(figsize=(10, 6))

plt.barh(
    df_top_words["word"][::-1],
    df_top_words["frequency"][::-1]
)

plt.title("15 palabras más frecuentes en Banking77")
plt.xlabel("Frecuencia")
plt.ylabel("Palabra")
plt.show()

### Interpretación de las palabras más frecuentes

El vocabulario más frecuente refleja claramente el dominio bancario del
conjunto de datos. Términos como `card`, `account`, `money`, `transfer`,
`payment`, `cash`, `exchange` y `atm` están relacionados directamente con
los productos y operaciones representados en Banking77.

Destaca especialmente la presencia de `not`, con **2,548 apariciones**, como
la segunda palabra más frecuente. Este resultado muestra la relevancia de las
expresiones negativas en las consultas y justifica su conservación durante
la limpieza exploratoria, ya que la negación puede modificar sustancialmente
el significado de una solicitud.

Por otra parte, la elevada frecuencia de términos como `card` no implica que
estos sean suficientes por sí solos para identificar una intención. Una misma
palabra puede aparecer en categorías relacionadas con entrega, activación,
funcionamiento, pagos o pérdida de tarjetas. Por tanto, el significado depende
también de las palabras que acompañan a estos términos y de su contexto.

Esta observación motiva el análisis de **bigramas y trigramas**, que permitirá
examinar combinaciones de palabras con mayor especificidad semántica.

## 1.5 Análisis de bigramas y trigramas

Las palabras individuales permiten identificar el dominio general de las
consultas, pero pueden resultar insuficientes para distinguir entre
intenciones semánticamente próximas. Por ello, se analizan las combinaciones
de dos y tres palabras consecutivas más frecuentes.

Los bigramas y trigramas permiten conservar una mayor cantidad de contexto
local y pueden revelar expresiones características de determinadas operaciones,
problemas o solicitudes bancarias.

In [ ]:
# ============================================================
# FUNCIÓN PARA OBTENER N-GRAMAS MÁS FRECUENTES
# ============================================================

def get_top_ngrams(texts, ngram_range, top_n=10):
    """
    Obtiene los n-gramas más frecuentes de una colección de textos.

    Parameters
    ----------
    texts : iterable
        Textos previamente normalizados.
    ngram_range : tuple
        Tamaño del n-grama. Ejemplo: (2, 2) para bigramas.
    top_n : int
        Número de resultados que se desean obtener.
    """

    vectorizer = CountVectorizer(ngram_range=ngram_range)

    ngram_matrix = vectorizer.fit_transform(texts)

    frequencies = np.asarray(
        ngram_matrix.sum(axis=0)
    ).ravel()

    ngrams = vectorizer.get_feature_names_out()

    df_ngrams = pd.DataFrame({
        "ngram": ngrams,
        "frequency": frequencies
    })

    return (
        df_ngrams
        .sort_values("frequency", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

In [ ]:
# ============================================================
# 10 BIGRAMAS Y 10 TRIGRAMAS MÁS FRECUENTES
# ============================================================

top_10_bigrams = get_top_ngrams(
    df_train["clean_text"],
    ngram_range=(2, 2),
    top_n=10
)

top_10_trigrams = get_top_ngrams(
    df_train["clean_text"],
    ngram_range=(3, 3),
    top_n=10
)

print("10 bigramas más frecuentes:")
display(top_10_bigrams)

print("\n10 trigramas más frecuentes:")
display(top_10_trigrams)

In [ ]:
# ============================================================
# VISUALIZACIÓN DE LOS 10 BIGRAMAS MÁS FRECUENTES
# ============================================================

plt.figure(figsize=(10, 6))

plt.barh(
    top_10_bigrams["ngram"][::-1],
    top_10_bigrams["frequency"][::-1]
)

plt.title("10 bigramas más frecuentes")
plt.xlabel("Frecuencia")
plt.ylabel("Bigrama")

plt.show()

In [ ]:
# ============================================================
# VISUALIZACIÓN DE LOS 10 TRIGRAMAS MÁS FRECUENTES
# ============================================================

plt.figure(figsize=(10, 6))

plt.barh(
    top_10_trigrams["ngram"][::-1],
    top_10_trigrams["frequency"][::-1]
)

plt.title("10 trigramas más frecuentes")
plt.xlabel("Frecuencia")
plt.ylabel("Trigrama")

plt.show()

### Interpretación de bigramas y trigramas

El análisis de n-gramas muestra que las combinaciones de palabras aportan
información más específica que los términos considerados de forma aislada.
Entre los bigramas más frecuentes aparecen expresiones claramente relacionadas
con operaciones bancarias, como `exchange rate`, `card payment`,
`virtual card` y `cash withdrawal`.

La conservación de las negaciones permite además identificar expresiones como
`card not` y `not working`. Esta información se vuelve todavía más específica
en los trigramas, donde aparecen combinaciones como `card not working`,
`card payment not` y `card not work`, asociadas a situaciones en las que el
usuario reporta un problema o una operación que no funciona como esperaba.

Otros trigramas, como `disposable virtual card`, `exchange rate wrong`,
`charged extra fee` y `direct debit payment`, representan conceptos bancarios
más concretos y potencialmente relacionados con intenciones específicas del
conjunto de datos.

Estos resultados evidencian que una palabra aislada puede ser compartida por
múltiples categorías, mientras que sus combinaciones aportan mayor contexto
semántico. Sin embargo, incluso consultas con vocabulario similar pueden
corresponder a intenciones diferentes, por lo que posteriormente será
importante analizar si esta proximidad semántica se refleja en los errores
del clasificador Transformer.

## 1.6 Nube de palabras

Como complemento al análisis de frecuencias, se genera una nube de palabras
a partir de las consultas normalizadas del conjunto de entrenamiento. Esta
visualización permite identificar de forma rápida los términos con mayor
presencia dentro del corpus.

In [ ]:
# ============================================================
# NUBE DE PALABRAS
# ============================================================

text_for_wordcloud = " ".join(df_train["clean_text"])

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    max_words=100,
    stopwords=set(),
    collocations=False,
    random_state=SEED
).generate(text_for_wordcloud)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Nube de palabras del conjunto de entrenamiento")
plt.show()

### Interpretación de la nube de palabras

La nube se generó desactivando la detección automática de colocaciones para
que el tamaño de cada término represente su frecuencia individual en el texto
normalizado. Las relaciones entre palabras se analizaron de forma independiente
mediante bigramas y trigramas.

La nube de palabras confirma visualmente los resultados del análisis de
frecuencias. Los términos con mayor presencia están relacionados directamente
con operaciones y servicios bancarios, destacando conceptos asociados con
tarjetas, cuentas, transferencias, pagos, efectivo, tipos de cambio y cajeros
automáticos. También destaca la elevada presencia de `not`, lo que evidencia
la frecuencia de expresiones negativas dentro de las consultas y respalda la
decisión de conservar las negaciones durante el preprocesamiento.

La elevada presencia de términos compartidos por diferentes tipos de consultas
refuerza la idea de que las palabras individuales no son suficientes para
distinguir todas las intenciones. La clasificación requiere interpretar el
contexto en el que dichos términos aparecen.

## 1.7 Distribución y balance de las clases

Se analiza la distribución de las 77 intenciones en los conjuntos de
entrenamiento y prueba con el objetivo de identificar posibles diferencias
en el número de muestras disponibles por categoría. Este análisis permitirá
evaluar posteriormente si la representación de las clases durante el
entrenamiento guarda relación con el desempeño obtenido por el clasificador.

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE LAS 77 CLASES
# ============================================================

class_distribution = (
    df_train["intent"]
    .value_counts()
    .sort_values(ascending=False)
)

print("Número de clases:", class_distribution.size)
print("Muestras de la clase con mayor representación:",
      class_distribution.max())
print("Muestras de la clase con menor representación:",
      class_distribution.min())
print("Promedio de muestras por clase:",
      round(class_distribution.mean(), 2))
print("Desviación estándar:",
      round(class_distribution.std(), 2))

display(class_distribution)

In [ ]:
# ============================================================
# CLASES CON MAYOR Y MENOR REPRESENTACIÓN
# ============================================================

imbalance_ratio = class_distribution.max() / class_distribution.min()

print(f"Relación máximo/mínimo: {imbalance_ratio:.2f}")

print("\n5 clases con mayor representación:")
display(class_distribution.head(5))

print("\n5 clases con menor representación:")
display(class_distribution.tail(5))

In [ ]:
# ============================================================
# VISUALIZACIÓN DE LA DISTRIBUCIÓN DE CLASES
# ============================================================

plt.figure(figsize=(12, 18))

plt.barh(
    class_distribution.index[::-1],
    class_distribution.values[::-1]
)

plt.axvline(
    class_distribution.mean(),
    linestyle="--",
    label=f"Promedio = {class_distribution.mean():.2f}"
)

plt.title("Distribución de las 77 clases en el conjunto de entrenamiento")
plt.xlabel("Número de muestras")
plt.ylabel("Intención")
plt.legend()

plt.show()

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE CLASES EN EL CONJUNTO DE PRUEBA
# ============================================================

test_class_distribution = (
    df_test["intent"]
    .value_counts()
    .sort_values(ascending=False)
)

print("Número de clases:", test_class_distribution.size)
print("Máximo de muestras por clase:", test_class_distribution.max())
print("Mínimo de muestras por clase:", test_class_distribution.min())
print("Promedio de muestras por clase:",
      round(test_class_distribution.mean(), 2))
print("Desviación estándar:",
      round(test_class_distribution.std(), 2))

### Interpretación de la distribución de clases

El conjunto de entrenamiento presenta diferencias apreciables en el número de
muestras disponibles por intención. Las 77 clases contienen en promedio
129.91 consultas, con una desviación estándar de 32.94. La clase con mayor
representación contiene 187 muestras, mientras que la de menor representación
contiene 35, lo que produce una relación máximo/mínimo de aproximadamente 5.34.

Entre las clases menos representadas se encuentran `contactless_not_working`
(35 muestras), `virtual_card_not_working` (41), `card_acceptance` (59),
`card_swallowed` (61) y `lost_or_stolen_card` (82). En contraste,
`card_payment_fee_charged` alcanza 187 muestras.

El conjunto de prueba presenta una distribución diferente: las 77 intenciones
contienen exactamente 40 consultas cada una, con una desviación estándar de
0. Esto proporciona una evaluación balanceada en cuanto al número de ejemplos
por categoría.

La diferencia de representación observada durante el entrenamiento plantea la
hipótesis de que las clases con menos ejemplos podrían presentar un desempeño
inferior. Sin embargo, el número de muestras no constituye necesariamente la
única fuente de dificultad, ya que el análisis previo mostró la existencia de
vocabulario y expresiones compartidas entre distintas intenciones.

Esta hipótesis será contrastada posteriormente mediante las métricas por clase
del modelo Transformer, analizando la relación entre el número de muestras de
entrenamiento y el F1 obtenido por cada intención, así como los patrones
observados en la matriz de confusión.

## 1.8 Conclusiones del análisis exploratorio

El análisis exploratorio muestra que Banking77 está compuesto principalmente
por consultas breves. En entrenamiento, las consultas contienen en promedio
11.95 palabras, con una mediana de 10, mientras que en prueba presentan una
media de 10.95 y una mediana de 9. Las distribuciones de longitud son similares
en ambas particiones, por lo que no se observan diferencias importantes entre
entrenamiento y prueba respecto al tamaño de los textos. Esta brevedad implica
que la identificación de la intención debe realizarse a partir de una cantidad
limitada de contexto.

El análisis léxico evidencia un vocabulario fuertemente relacionado con el
dominio bancario. Palabras como `card`, `account`, `money`, `transfer`,
`payment` y `cash` aparecen con elevada frecuencia. También destaca `not`,
con 2,548 apariciones, lo que muestra la relevancia de las expresiones
negativas dentro de las consultas y justifica su conservación durante el
preprocesamiento exploratorio.

Sin embargo, las palabras individuales proporcionan información limitada para
distinguir las 77 intenciones. Los bigramas y trigramas muestran que
combinaciones como `exchange rate`, `card payment`, `virtual card`,
`cash withdrawal`, `card not working` o `charged extra fee` aportan mayor
especificidad semántica. Esto sugiere que la clasificación depende no solo de
la presencia de determinadas palabras, sino también del contexto y de las
relaciones entre ellas.

Respecto a la distribución de las clases, el conjunto de entrenamiento no está
uniformemente balanceado. El número de ejemplos varía entre 35 y 187 consultas
por intención, con una relación máximo/mínimo de 5.34. En contraste, el
conjunto de prueba está perfectamente balanceado, con 40 consultas para cada
una de las 77 categorías.

A partir de estos resultados se plantean dos hipótesis para el análisis del
clasificador. En primer lugar, las intenciones con menor representación durante
el entrenamiento podrían presentar un desempeño inferior. En segundo lugar,
las intenciones que comparten vocabulario o expresiones semánticamente próximas
podrían generar confusiones incluso cuando disponen de un número suficiente de
ejemplos.

Estas hipótesis se contrastarán durante la evaluación de DistilRoBERTa mediante
las métricas por clase, la relación entre el número de muestras de entrenamiento
y el F1, y el análisis de la matriz de confusión.

# 2. Clasificación de intenciones con DistilRoBERTa

En esta sección se realizará el fine-tuning del modelo preentrenado
`distilroberta-base` para clasificar las consultas de Banking77 entre sus
77 posibles intenciones.

Para entrenar y evaluar el modelo se utilizarán tres particiones. El conjunto
de entrenamiento original se dividirá de forma estratificada en entrenamiento
y validación, mientras que la partición `test` proporcionada por Banking77 se
mantendrá sin modificaciones para la evaluación final.

El texto utilizado por el Transformer será la columna original `text`, sin
aplicar la limpieza empleada durante el EDA. De esta forma, el tokenizer
preentrenado podrá procesar las consultas conservando su estructura y contexto
lingüístico.

Se utiliza DistilRoBERTa como modelo base para la clasificación. Este modelo es
una versión destilada de RoBERTa que conserva su arquitectura general con un
menor número de capas, reduciendo los requerimientos computacionales
(Sanh et al., 2019; Liu et al., 2019).

## 2.1 Preparación de los datos

Para realizar el fine-tuning de DistilRoBERTa se utilizarán tres particiones:
entrenamiento, validación y prueba. El conjunto de entrenamiento original de
Banking77 se dividirá en un 90 % para entrenamiento y un 10 % para validación,
mientras que el conjunto de prueba proporcionado originalmente por el dataset
se conservará sin modificaciones para la evaluación final.

La división entre entrenamiento y validación se realizará de forma
estratificada utilizando las etiquetas de las 77 intenciones. De esta manera
se busca conservar, en ambas particiones, una distribución proporcional a la
existente en el conjunto de entrenamiento original.

El modelo utilizará las consultas originales almacenadas en la columna `text`.
La versión normalizada `clean_text`, generada durante el análisis exploratorio,
no se utilizará para el entrenamiento, ya que DistilRoBERTa dispone de su
propio tokenizer y puede aprovechar información contextual presente en el
texto original.

In [ ]:
# ============================================================
# DIVISIÓN TRAIN / VALIDATION / TEST
# ============================================================

# Índices del conjunto de entrenamiento original
indices = np.arange(len(dataset["train"]))

# División estratificada 90 % entrenamiento / 10 % validación
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=SEED,
    stratify=dataset["train"]["label"]
)

# Crear las particiones mediante los índices obtenidos
train_dataset = dataset["train"].select(train_idx)
val_dataset = dataset["train"].select(val_idx)

# El conjunto de prueba original se conserva sin modificaciones
test_dataset = dataset["test"]

print("Muestras de entrenamiento:", len(train_dataset))
print("Muestras de validación:", len(val_dataset))
print("Muestras de prueba:", len(test_dataset))

In [ ]:
# ============================================================
# VERIFICACIÓN DE LA DIVISIÓN ESTRATIFICADA
# ============================================================

train_labels = pd.Series(train_dataset["label"])
val_labels = pd.Series(val_dataset["label"])

split_check = pd.DataFrame({
    "train": train_labels.value_counts().sort_index(),
    "validation": val_labels.value_counts().sort_index()
})

split_check["total"] = (
    split_check["train"] +
    split_check["validation"]
)

split_check["validation_pct"] = (
    split_check["validation"] /
    split_check["total"] * 100
)

print("Número de clases en entrenamiento:",
      split_check["train"].count())

print("Número de clases en validación:",
      split_check["validation"].count())

print(
    "Porcentaje medio destinado a validación:",
    round(split_check["validation_pct"].mean(), 2),
    "%"
)

display(split_check.head(10))

### Verificación de la división estratificada

La división generó 9,002 muestras para entrenamiento y 1,001 para validación,
manteniendo las 3,080 muestras del conjunto de prueba original para la
evaluación final.

Las 77 intenciones están presentes tanto en entrenamiento como en validación.
En promedio, el 9.99 % de las muestras de cada clase fue asignado al conjunto
de validación. Las pequeñas variaciones respecto al 10 % se deben al número
entero de observaciones disponible en cada categoría.

De esta manera, la partición de validación conserva aproximadamente la
distribución de clases del conjunto de entrenamiento original y puede
utilizarse para evaluar el comportamiento del modelo durante el fine-tuning
sin utilizar el conjunto de prueba.

## 2.2 Tokenización con DistilRoBERTa

DistilRoBERTa utiliza un tokenizer basado en subpalabras para transformar las
consultas de texto en secuencias de identificadores que pueden ser procesadas
por el modelo.

Antes de establecer una longitud máxima para las secuencias, se analizará la
cantidad de tokens generados por el tokenizer sobre el conjunto de
entrenamiento. De esta forma, el valor utilizado durante el fine-tuning se
seleccionará a partir de las características reales de los datos y no de forma
arbitraria.

Para este proceso se utilizarán las consultas originales de Banking77, sin
aplicar la normalización empleada exclusivamente durante el análisis
exploratorio.

In [ ]:
# ============================================================
# TOKENIZER DE DISTILROBERTA
# ============================================================

MODEL_NAME = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Modelo:", MODEL_NAME)
print("Tamaño del vocabulario:", tokenizer.vocab_size)
print("Longitud máxima admitida:", tokenizer.model_max_length)

In [ ]:
# ============================================================
# EJEMPLO DE TOKENIZACIÓN
# ============================================================

example = train_dataset[0]

text = example["text"]
label = example["label"]

encoded = tokenizer(
    text,
    truncation=True,
    max_length=128
)

tokens = tokenizer.convert_ids_to_tokens(
    encoded["input_ids"]
)

print("Consulta original:")
print(text)

print("\nLabel numérico:", label)
print("Intención:", label_names[label])

print("\nTokens:")
print(tokens)

print("\nInput IDs:")
print(encoded["input_ids"])

print("\nAttention mask:")
print(encoded["attention_mask"])

### Ejemplo de tokenización

El ejemplo permite observar cómo el tokenizer de DistilRoBERTa transforma una
consulta original en una secuencia de tokens y posteriormente en identificadores
numéricos (`input_ids`) que pueden ser procesados por el modelo.

La `attention_mask` indica qué posiciones corresponden a tokens válidos de la
secuencia. En este ejemplo todavía no se observa padding, ya que este se
incorporará dinámicamente al formar los batches durante el entrenamiento.

También pueden aparecer tokens especiales añadidos automáticamente por el
tokenizer para representar los límites de la secuencia. De esta manera, el
modelo no recibe directamente palabras, sino una representación numérica
compatible con la arquitectura Transformer.

In [ ]:
# ============================================================
# ANÁLISIS DE LONGITUD EN TOKENS
# ============================================================

token_lengths = [
    len(tokenizer(text, truncation=False)["input_ids"])
    for text in train_dataset["text"]
]

token_lengths = np.array(token_lengths)

print("Número de consultas analizadas:", len(token_lengths))
print("Mínimo:", token_lengths.min())
print("Media:", round(token_lengths.mean(), 2))
print("Mediana:", round(np.median(token_lengths), 2))
print("Percentil 95:", round(np.percentile(token_lengths, 95), 2))
print("Percentil 99:", round(np.percentile(token_lengths, 99), 2))
print("Máximo:", token_lengths.max())

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE LA LONGITUD EN TOKENS
# ============================================================

plt.figure(figsize=(10, 5))

plt.hist(token_lengths, bins=40)

plt.axvline(
    np.percentile(token_lengths, 95),
    linestyle="--",
    label=f"P95 = {np.percentile(token_lengths, 95):.0f}"
)

plt.axvline(
    np.percentile(token_lengths, 99),
    linestyle=":",
    label=f"P99 = {np.percentile(token_lengths, 99):.0f}"
)

plt.title("Distribución de la longitud de las consultas en tokens")
plt.xlabel("Número de tokens")
plt.ylabel("Número de consultas")
plt.legend()

plt.show()

### Análisis de la longitud en tokens

Las consultas de entrenamiento presentan una longitud media de 15.79 tokens
y una mediana de 13 tokens. El 95 % de las consultas contiene como máximo
36 tokens y el 99 % no supera los 50. La consulta más extensa alcanza
96 tokens.

A partir de estos resultados se establece una longitud máxima de 128 tokens.
Este valor supera la longitud máxima observada en el conjunto de entrenamiento,
por lo que permite conservar completamente las consultas analizadas y deja un
margen adicional para procesar las particiones de validación y prueba.

Utilizar una longitud considerablemente mayor, como el máximo admitido por el
modelo, incrementaría innecesariamente el costo computacional del padding sin
aportar información adicional para la mayoría de las consultas.

In [ ]:
# Longitud máxima utilizada durante la tokenización
MAX_LENGTH = 128

### Tokenización de las particiones

Las consultas de entrenamiento, validación y prueba se tokenizan utilizando
el tokenizer de `distilroberta-base`, estableciendo 128 tokens como longitud
máxima y activando el truncamiento para garantizar que ninguna secuencia
supere dicho límite.

No se aplica padding hasta la longitud máxima durante esta etapa. En su lugar,
se utilizará padding dinámico durante el entrenamiento, de manera que las
secuencias de cada batch se rellenen únicamente hasta la longitud de la
consulta más extensa de ese batch. Esta estrategia reduce el procesamiento
innecesario de tokens de padding, especialmente considerando la corta longitud
media de las consultas de Banking77.

In [ ]:
# ============================================================
# FUNCIÓN DE TOKENIZACIÓN
# ============================================================

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
# ============================================================
# TOKENIZACIÓN DE TRAIN, VALIDATION Y TEST
# ============================================================

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_batch,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_batch,
    batched=True
)

print(tokenized_train)

In [ ]:
# ============================================================
# PADDING DINÁMICO
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator configurado para padding dinámico.")

### Resultado de la tokenización

La tokenización se completó correctamente para las tres particiones. Cada
consulta dispone ahora de `input_ids`, que representan los identificadores de
los tokens utilizados por DistilRoBERTa, y `attention_mask`, que permite al
modelo distinguir los tokens reales de los que posteriormente se incorporen
como padding.

El padding se realizará dinámicamente durante la formación de los batches,
evitando rellenar todas las consultas hasta los 128 tokens definidos como
longitud máxima.

## 2.3 Configuración del modelo DistilRoBERTa

Se utilizará el modelo preentrenado `distilroberta-base` adaptándolo a una
tarea de clasificación multiclase con 77 categorías, correspondientes a las
intenciones definidas en Banking77.

Para mantener una correspondencia explícita entre las etiquetas numéricas del
dataset y sus nombres, se crearán los diccionarios `id2label` y `label2id`.
Estos serán incorporados a la configuración del modelo, facilitando la
posterior interpretación de las predicciones y el análisis de errores.

La arquitectura preentrenada proporciona las representaciones lingüísticas
iniciales, mientras que la cabeza de clasificación se ajustará durante el
fine-tuning para resolver específicamente la tarea de clasificación de
intenciones.

In [ ]:
# ============================================================
# CONFIGURACIÓN DE LAS ETIQUETAS
# ============================================================

NUM_LABELS = len(label_names)

id2label = {
    idx: label
    for idx, label in enumerate(label_names)
}

label2id = {
    label: idx
    for idx, label in enumerate(label_names)
}

print("Número de etiquetas:", NUM_LABELS)

print("\nPrimeras 10 correspondencias:")
for idx in range(10):
    print(f"{idx:2d} -> {id2label[idx]}")

In [ ]:
# ============================================================
# MODELO DISTILROBERTA PARA CLASIFICACIÓN
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

print("Modelo cargado:", MODEL_NAME)
print("Número de clases configuradas:", model.config.num_labels)

In [ ]:
# ============================================================
# VERIFICACIÓN DE LA CONFIGURACIÓN
# ============================================================

assert model.config.num_labels == 77
assert len(model.config.id2label) == 77
assert len(model.config.label2id) == 77

print("Configuración verificada correctamente.")
print("Salida esperada del clasificador: 77 clases.")

### Verificación de la configuración del modelo

DistilRoBERTa se cargó correctamente con una salida configurada para las 77
intenciones de Banking77. Durante la carga, la librería informó que los
parámetros correspondientes a la cabeza de clasificación fueron inicializados
para la nueva tarea.

Este comportamiento es esperado, ya que el checkpoint `distilroberta-base`
proporciona las representaciones lingüísticas preentrenadas, pero no dispone
de una cabeza previamente ajustada para las 77 categorías específicas de
Banking77. Estos parámetros serán aprendidos durante el proceso de fine-tuning.

## 2.4 Métricas de evaluación

El desempeño del clasificador se evaluará mediante `accuracy`, precisión,
recall y F1. Debido a que el problema contiene 77 intenciones, se utilizará
promedio macro para precisión, recall y F1, de manera que cada clase contribuya
con el mismo peso al resultado global, independientemente de su número de
muestras.

La `accuracy` permitirá medir la proporción total de consultas correctamente
clasificadas, mientras que las métricas macro permitirán observar el
comportamiento global considerando por igual todas las intenciones.

Durante el fine-tuning estas métricas se calcularán sobre el conjunto de
validación. Posteriormente, sobre el conjunto de prueba, se realizará además
un análisis detallado por clase para identificar las intenciones con mejor y
peor desempeño.

In [ ]:
# ============================================================
# MÉTRICAS DE EVALUACIÓN
# ============================================================

def compute_metrics(eval_pred):
    """
    Calcula las métricas globales utilizadas durante la
    evaluación del modelo.
    """

    logits, labels = eval_pred

    # Seleccionar la clase con mayor puntuación
    predictions = np.argmax(logits, axis=-1)

    # Accuracy global
    accuracy = accuracy_score(labels, predictions)

    # Precision, recall y F1 con promedio macro
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

## 2.5 Configuración del fine-tuning

El fine-tuning de DistilRoBERTa se realizará durante tres épocas utilizando
una tasa de aprendizaje de `2e-5`, un valor habitual para el ajuste de modelos
Transformer preentrenados. Se empleará un tamaño de batch de 16 muestras
durante el entrenamiento y 32 durante la evaluación.

Se aplicará `weight_decay=0.01` como mecanismo de regularización y se utilizará
precisión mixta FP16 para aprovechar la GPU NVIDIA Tesla T4 y reducir el
consumo de memoria durante el entrenamiento.

El desempeño del modelo se evaluará al finalizar cada época. Asimismo, se
guardará un checkpoint por época y al terminar el entrenamiento se recuperará
automáticamente el modelo que haya obtenido el mayor F1-macro sobre el conjunto
de validación.

El conjunto de prueba no participará en este proceso y se utilizará únicamente
después de seleccionar el modelo final.

In [ ]:
# ============================================================
# 2.5 CONFIGURACIÓN DEL FINE-TUNING
# ============================================================

training_args = TrainingArguments(
    output_dir="./distilroberta_banking77",

    # Optimización
    learning_rate=2e-5,
    weight_decay=0.01,

    # Tamaños de batch
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Número de épocas
    num_train_epochs=3,

    # Evaluación y checkpoints
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    # Selección del mejor modelo
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # GPU T4
    fp16=True,

    # Seguimiento
    logging_strategy="steps",
    logging_steps=100,
    report_to="none",

    # Reproducibilidad
    seed=SEED,
    data_seed=SEED
)

print("Configuración del entrenamiento creada correctamente.")

In [ ]:
# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer configurado correctamente.")

In [ ]:
print("Modelo:", MODEL_NAME)
print("Clases:", model.config.num_labels)
print("Train:", len(tokenized_train))
print("Validation:", len(tokenized_val))
print("Test reservado:", len(tokenized_test))

print("\nConfiguración:")
print("Learning rate:", training_args.learning_rate)
print("Batch train:", training_args.per_device_train_batch_size)
print("Batch eval:", training_args.per_device_eval_batch_size)
print("Épocas:", training_args.num_train_epochs)
print("FP16:", training_args.fp16)
print("Métrica mejor modelo:", training_args.metric_for_best_model)

## 2.6 Fine-tuning de DistilRoBERTa

Con el modelo, los datos y las métricas configurados, se realiza el fine-tuning
de DistilRoBERTa utilizando exclusivamente los conjuntos de entrenamiento y
validación.

Durante las tres épocas se registrarán la pérdida de entrenamiento, la pérdida
de validación y las métricas definidas anteriormente. Al finalizar, el
`Trainer` restaurará automáticamente el checkpoint que haya obtenido el mayor
F1-macro sobre el conjunto de validación.

El conjunto de prueba permanece excluido de este proceso y se utilizará
posteriormente para evaluar el desempeño final del modelo seleccionado.

In [ ]:
# ============================================================
# 2.6 FINE-TUNING DE DISTILROBERTA
# ============================================================

train_result = trainer.train()

In [ ]:
# ============================================================
# RESUMEN DEL ENTRENAMIENTO
# ============================================================

print("Entrenamiento finalizado.")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Tiempo total: {train_result.metrics['train_runtime']:.2f} s")
print(
    f"Muestras procesadas por segundo: "
    f"{train_result.metrics['train_samples_per_second']:.2f}"
)

### Análisis del proceso de entrenamiento

El desempeño del modelo mejoró progresivamente durante las tres épocas de
fine-tuning. La pérdida de entrenamiento disminuyó de 2.0869 en la primera
época a 0.8585 en la tercera, mientras que la pérdida de validación se redujo
de 1.5271 a 0.7357.

Esta reducción simultánea de las pérdidas de entrenamiento y validación estuvo
acompañada por una mejora de las métricas. La accuracy de validación aumentó
de 76.62 % a 87.31 %, mientras que el F1-macro pasó de 71.63 % a 85.29 %.

La tercera época obtuvo el mayor F1-macro y fue seleccionada automáticamente
como el mejor checkpoint de acuerdo con el criterio establecido previamente.

Dentro de las tres épocas evaluadas no se observan señales evidentes de
sobreajuste, ya que la pérdida de validación continúa disminuyendo mientras
las métricas de validación siguen mejorando. Tampoco se observan evidencias
de underfitting al finalizar el entrenamiento, dado que el modelo alcanza un
desempeño elevado sobre datos de validación no utilizados para actualizar sus
parámetros.

Estos resultados corresponden únicamente al conjunto de validación. La
capacidad de generalización del modelo se evaluará a continuación utilizando
por primera vez el conjunto de prueba reservado.

## 2.7 Evaluación final sobre el conjunto de prueba

Una vez finalizado el fine-tuning y seleccionado el mejor checkpoint mediante
el conjunto de validación, se evalúa el modelo sobre las 3,080 consultas del
conjunto de prueba.

Esta partición no fue utilizada durante el entrenamiento, la selección de
hiperparámetros ni la elección del mejor checkpoint, por lo que proporciona
una estimación independiente del desempeño final del clasificador.

In [ ]:
# ============================================================
# 2.7 EVALUACIÓN FINAL SOBRE TEST
# ============================================================

test_metrics = trainer.evaluate(tokenized_test)

print("RESULTADOS SOBRE EL CONJUNTO DE PRUEBA")
print("=" * 50)

print(f"Loss:            {test_metrics['eval_loss']:.4f}")
print(f"Accuracy:        {test_metrics['eval_accuracy']:.4f}")
print(f"Precision macro: {test_metrics['eval_precision_macro']:.4f}")
print(f"Recall macro:    {test_metrics['eval_recall_macro']:.4f}")
print(f"F1 macro:        {test_metrics['eval_f1_macro']:.4f}")

### Resultados globales sobre el conjunto de prueba

El modelo alcanzó una accuracy de **86.79 %** sobre las 3,080 consultas del
conjunto de prueba. Asimismo, obtuvo una precisión macro de **88.38 %**, un
recall macro de **86.79 %** y un F1-macro de **86.02 %**.

Los resultados son similares a los obtenidos sobre el conjunto de validación.
La accuracy disminuyó únicamente de 87.31 % a 86.79 %, mientras que el
F1-macro pasó de 85.29 % a 86.02 %. Esta proximidad entre ambos conjuntos
sugiere que el modelo mantiene un comportamiento estable sobre datos que no
participaron en el proceso de entrenamiento.

Sin embargo, las métricas globales no permiten determinar si el desempeño es
uniforme entre las 77 intenciones. Por este motivo, a continuación se
analizarán las métricas por clase y los patrones de confusión del modelo.

## 2.8 Predicciones y métricas por clase

Una vez evaluado el desempeño global de DistilRoBERTa, se obtendrán las
predicciones individuales sobre las 3,080 consultas del conjunto de prueba
para analizar el comportamiento del modelo en cada una de las 77 intenciones.

A partir de estas predicciones se calcularán la precisión, el recall y el F1
por clase, así como el número de predicciones correctas e incorrectas para
cada intención.

Este análisis permitirá identificar las siete intenciones con mejor desempeño
y las siete que presentan el mayor número de errores. Posteriormente, estos
resultados se utilizarán para estudiar los principales patrones de confusión
del modelo y contrastarlos con las hipótesis planteadas durante el análisis
exploratorio de datos.

In [ ]:
# ============================================================
# PREDICCIONES SOBRE EL CONJUNTO DE PRUEBA
# ============================================================

test_predictions = trainer.predict(tokenized_test)

logits = test_predictions.predictions
y_true = test_predictions.label_ids
y_pred = np.argmax(logits, axis=-1)

correct_predictions = (y_true == y_pred).sum()
incorrect_predictions = (y_true != y_pred).sum()

print("Total de consultas:", len(y_true))
print("Predicciones correctas:", correct_predictions)
print("Predicciones incorrectas:", incorrect_predictions)
print(
    "Accuracy calculada:",
    round(correct_predictions / len(y_true), 4)
)

In [ ]:
# ============================================================
# MÉTRICAS POR CLASE
# ============================================================

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_LABELS)),
    target_names=label_names,
    output_dict=True,
    zero_division=0
)

class_metrics = pd.DataFrame(report).T.iloc[:NUM_LABELS].copy()

class_metrics = class_metrics.reset_index().rename(
    columns={"index": "intent"}
)

class_metrics["support"] = class_metrics["support"].astype(int)

# Número de predicciones correctas e incorrectas por intención
class_metrics["correct"] = np.round(
    class_metrics["recall"] * class_metrics["support"]
).astype(int)

class_metrics["errors"] = (
    class_metrics["support"] - class_metrics["correct"]
)

display(class_metrics.head(10))

In [ ]:
# ============================================================
# INTENCIONES CON MEJOR DESEMPEÑO
# ============================================================

best_7 = (
    class_metrics
    .sort_values(
        by=["f1-score", "correct", "precision"],
        ascending=False
    )
    .head(7)
)

print("7 INTENCIONES CON MAYOR F1")
display(
    best_7[
        ["intent", "precision", "recall",
         "f1-score", "correct", "errors"]
    ]
)

In [ ]:
# ============================================================
# INTENCIONES CON MAYOR NÚMERO DE ERRORES
# ============================================================

worst_7 = (
    class_metrics
    .sort_values(
        by=["errors", "f1-score"],
        ascending=[False, True]
    )
    .head(7)
)

print("7 INTENCIONES CON MAYOR NÚMERO DE ERRORES")
display(
    worst_7[
        ["intent", "precision", "recall",
         "f1-score", "correct", "errors"]
    ]
)

### Análisis de las métricas por clase

El análisis por intención muestra que el desempeño de DistilRoBERTa no es
uniforme entre las 77 categorías. Varias intenciones presentan un desempeño
muy elevado. `edit_personal_details` y `verify_top_up` alcanzaron un F1 de
1.00, clasificando correctamente sus 40 consultas de prueba. Otras
intenciones, como `card_about_to_expire`, `verify_source_of_funds`,
`age_limit`, `apple_pay_or_google_pay` y `passcode_forgotten`, también
obtuvieron un recall de 1.00 y, por tanto, clasificaron correctamente las
40 consultas correspondientes.

En contraste, algunas intenciones presentan dificultades importantes. El caso
más significativo es `virtual_card_not_working`, donde solamente 1 de las
40 consultas fue clasificada correctamente, produciendo 39 errores y un
F1 de aproximadamente 0.049. También se observan cantidades elevadas de
errores en `card_swallowed` (27), `card_acceptance` (25),
`why_verify_identity` (21), `topping_up_by_card` (17),
`get_disposable_virtual_card` (13) y `pending_transfer` (13).

Estos resultados muestran que las métricas globales pueden ocultar diferencias
considerables entre categorías. Aunque el modelo alcanza una accuracy global
de 86.79 %, existen intenciones concretas en las que su capacidad de
discriminación es considerablemente menor.

Para determinar las posibles causas de estas diferencias será necesario
analizar hacia qué categorías se están desviando las predicciones incorrectas
y contrastar estos patrones con la cantidad de ejemplos disponibles durante
el entrenamiento y con la similitud semántica observada durante el análisis
exploratorio.

## 2.9 Análisis de las causas de error

A partir de las diferencias observadas en el desempeño por clase, se analizarán
dos posibles factores relacionados con los errores del clasificador.

En primer lugar, se estudiará la relación entre el número de muestras
disponibles para cada intención durante el entrenamiento y el F1 obtenido sobre
el conjunto de prueba. Este análisis permitirá contrastar la hipótesis planteada
durante el EDA acerca de si las clases con menor representación tienden a
presentar un desempeño inferior.

En segundo lugar, se analizarán los principales patrones de confusión del
modelo para determinar hacia qué intenciones se desvían las predicciones
incorrectas. Esto permitirá estudiar si los errores se concentran entre
categorías con vocabulario o significado semánticamente próximo, como sugería
el análisis de palabras, bigramas y trigramas realizado previamente.

### 2.9.1 Relación entre el número de muestras de entrenamiento y el F1

Durante el análisis exploratorio se observó que el conjunto de entrenamiento
no presenta una distribución uniforme entre las 77 intenciones. El número de
muestras por clase varía entre 35 y 187, por lo que se planteó la hipótesis de
que las categorías con menor representación podrían presentar un desempeño
inferior.

Para contrastar esta hipótesis, se relaciona el número de muestras disponibles
para cada intención en el conjunto de entrenamiento con el F1 obtenido sobre
el conjunto de prueba.

Además de la inspección gráfica, se utiliza el coeficiente de correlación de
Spearman para evaluar si existe una asociación monotónica entre ambas
variables. Esta medida resulta adecuada porque no requiere asumir una relación
lineal entre el número de muestras y el desempeño del clasificador.

In [ ]:
# ============================================================
# RELACIÓN ENTRE MUESTRAS DE ENTRENAMIENTO Y F1
# ============================================================

train_counts = (
    df_train["intent"]
    .value_counts()
    .rename("train_samples")
)

class_metrics_analysis = class_metrics.merge(
    train_counts,
    left_on="intent",
    right_index=True
)

display(
    class_metrics_analysis[
        ["intent", "train_samples", "f1-score", "errors"]
    ].sort_values("f1-score").head(10)
)

In [ ]:
# ============================================================
# MUESTRAS DE ENTRENAMIENTO VS. F1 POR CLASE
# ============================================================

plt.figure(figsize=(9, 6))

plt.scatter(
    class_metrics_analysis["train_samples"],
    class_metrics_analysis["f1-score"],
    alpha=0.7
)

plt.xlabel("Número de muestras de entrenamiento")
plt.ylabel("F1 en el conjunto de prueba")
plt.title("Relación entre muestras de entrenamiento y F1 por intención")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ============================================================
# CORRELACIÓN DE SPEARMAN
# ============================================================

rho, p_value = spearmanr(
    class_metrics_analysis["train_samples"],
    class_metrics_analysis["f1-score"]
)

print(f"Correlación de Spearman: {rho:.4f}")
print(f"p-value: {p_value:.4f}")

#### Interpretación

El diagrama de dispersión no muestra una tendencia claramente definida entre
el número de muestras de entrenamiento y el F1 obtenido por cada intención.
Aunque algunas de las categorías con menor representación presentan valores
bajos de F1, también existen clases con cantidades similares de ejemplos que
alcanzan un desempeño elevado.

La correlación de Spearman obtenida fue de **ρ = 0.1514**, lo que representa
una asociación positiva débil entre ambas variables. Además, el valor
**p = 0.1887** es superior al nivel de significancia convencional de 0.05,
por lo que la asociación observada no resulta estadísticamente significativa.

Por tanto, los resultados no proporcionan evidencia suficiente para afirmar
que un mayor número de muestras de entrenamiento esté asociado, de forma
general y monotónica, con un mayor F1 entre las 77 intenciones.

El caso de `virtual_card_not_working` resulta especialmente relevante, ya que
dispone únicamente de 41 muestras de entrenamiento y obtiene un F1 de 0.0488.
Sin embargo, este caso individual no permite atribuir su bajo desempeño
exclusivamente a la cantidad de datos. Por ejemplo, otras intenciones con
cantidades relativamente bajas de muestras alcanzan valores de F1
considerablemente superiores.

En consecuencia, la primera hipótesis planteada durante el EDA no queda
respaldada como explicación general del desempeño por clase. Esto sugiere que
es necesario considerar otros factores, particularmente la similitud semántica
entre intenciones, que se analizará a continuación mediante los patrones de
confusión del modelo.

### 2.9.2 Principales patrones de confusión

Además de la cantidad de ejemplos disponibles durante el entrenamiento, los
errores pueden estar relacionados con la similitud existente entre diferentes
intenciones.

Para estudiar este comportamiento se analizarán las predicciones incorrectas
de las siete categorías con mayor número de errores, identificando las
intenciones hacia las que el modelo desvía con mayor frecuencia sus
predicciones.

In [ ]:
# ============================================================
# DATAFRAME DE PREDICCIONES
# ============================================================

predictions_df = pd.DataFrame({
    "text": test_dataset["text"],
    "true_id": y_true,
    "pred_id": y_pred
})

predictions_df["true_intent"] = predictions_df["true_id"].map(id2label)
predictions_df["predicted_intent"] = predictions_df["pred_id"].map(id2label)

errors_df = predictions_df[
    predictions_df["true_id"] != predictions_df["pred_id"]
].copy()

print("Total de errores:", len(errors_df))

display(
    errors_df[
        ["text", "true_intent", "predicted_intent"]
    ].head(10)
)

In [ ]:
# ============================================================
# PRINCIPALES CONFUSIONES DE LAS 7 CLASES CON MÁS ERRORES
# ============================================================

worst_intents = worst_7["intent"].tolist()

for intent in worst_intents:

    intent_errors = errors_df[
        errors_df["true_intent"] == intent
    ]

    confusions = (
        intent_errors["predicted_intent"]
        .value_counts()
        .head(5)
    )

    print(f"\nINTENCIÓN REAL: {intent}")
    print("-" * 60)
    print(confusions)

### Interpretación de las causas de error

La relación entre el número de muestras disponibles durante el entrenamiento
y el F1 obtenido por cada intención presenta una correlación de Spearman de
**ρ = 0.1514**, con un valor **p = 0.1887**. La asociación observada es
positiva pero débil y no resulta estadísticamente significativa al nivel
convencional de 0.05.

Por tanto, los resultados no proporcionan evidencia suficiente para sostener
que disponer de un mayor número de ejemplos de entrenamiento esté asociado,
de forma general y monotónica, con un mayor F1 entre las 77 intenciones. Esto
no implica que la cantidad de ejemplos sea irrelevante para clases concretas,
sino que no parece explicar por sí sola las diferencias de desempeño
observadas en el conjunto completo de categorías.

El análisis de las predicciones incorrectas muestra, en cambio, patrones de
confusión concentrados entre intenciones semánticamente relacionadas. El caso
más evidente corresponde a `virtual_card_not_working`: de sus 39 errores,
18 fueron clasificados como `getting_virtual_card` y 17 como
`get_disposable_virtual_card`. Por tanto, 35 de sus 39 errores se concentran
en solamente dos intenciones también relacionadas con tarjetas virtuales.

Un comportamiento similar aparece en `why_verify_identity`, cuyos 21 errores
se distribuyen únicamente entre `verify_my_identity` (16) y
`unable_to_verify_identity` (5). Asimismo, `card_swallowed` se confunde
principalmente con `declined_cash_withdrawal` y `topping_up_by_card` presenta
errores hacia otras categorías relacionadas con operaciones de recarga.

Estos patrones son coherentes con los resultados del análisis exploratorio.
Palabras como `card`, `transfer`, `payment` o `top` aparecen en numerosas
consultas y los n-gramas permiten aportar mayor contexto, pero diferentes
intenciones pueden continuar compartiendo vocabulario y conceptos muy
similares. En consecuencia, los resultados sugieren que la proximidad
semántica entre determinadas categorías constituye una fuente importante de
errores para el clasificador.

De las dos hipótesis planteadas durante el EDA, la primera, relacionada con la
cantidad de muestras de entrenamiento, no presenta una asociación general
estadísticamente significativa con el F1. La segunda, relacionada con la
similitud semántica entre intenciones, sí es consistente descriptivamente con
los patrones de confusión observados, aunque este análisis no establece por sí
solo una relación causal.

## 2.10 Matriz de confusión

Para complementar las métricas por clase se construirá la matriz de confusión
del conjunto de prueba. Esta representación permite observar simultáneamente
las etiquetas reales y las predicciones realizadas por DistilRoBERTa,
facilitando la identificación de patrones sistemáticos de error.

Debido a que Banking77 contiene 77 intenciones, se utilizarán dos
visualizaciones complementarias. La primera mostrará la matriz completa para
examinar el comportamiento general del clasificador. Posteriormente se
construirá una matriz enfocada en las intenciones con mayor número de errores
y en las categorías con las que se confunden con mayor frecuencia.

Esta segunda representación permitirá interpretar con mayor claridad las
confusiones detectadas en el análisis anterior.

In [ ]:
# ============================================================
# 2.10 MATRIZ DE CONFUSIÓN GLOBAL
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_LABELS))
)

print("Dimensiones de la matriz:", cm.shape)
print("Total de observaciones:", cm.sum())
print("Predicciones correctas:", np.trace(cm))
print("Predicciones incorrectas:", cm.sum() - np.trace(cm))

In [ ]:
# ============================================================
# VISUALIZACIÓN DE LA MATRIZ GLOBAL
# ============================================================

plt.figure(figsize=(18, 16))

sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=5,
    yticklabels=5,
    cbar=True
)

plt.title("Matriz de confusión global - DistilRoBERTa")
plt.xlabel("Clase predicha")
plt.ylabel("Clase real")

plt.show()

In [ ]:
# ============================================================
# CLASES PARA LA MATRIZ DE CONFUSIÓN ENFOCADA
# ============================================================

focus_intents = set(worst_7["intent"])

for intent in worst_7["intent"]:
    top_confusions = (
        errors_df.loc[
            errors_df["true_intent"] == intent,
            "predicted_intent"
        ]
        .value_counts()
        .head(2)
        .index
    )

    focus_intents.update(top_confusions)

# Mantener el orden original de Banking77
focus_intents = [
    intent for intent in label_names
    if intent in focus_intents
]

focus_ids = [
    label2id[intent]
    for intent in focus_intents
]

print("Número de intenciones seleccionadas:", len(focus_intents))

for intent in focus_intents:
    print(intent)

In [ ]:
# ============================================================
# MATRIZ DE CONFUSIÓN ENFOCADA
# ============================================================

cm_focus = confusion_matrix(
    y_true,
    y_pred,
    labels=focus_ids
)

plt.figure(figsize=(14, 11))

sns.heatmap(
    cm_focus,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=focus_intents,
    yticklabels=focus_intents,
    cbar=True
)

plt.title("Matriz de confusión enfocada en las clases problemáticas")
plt.xlabel("Intención predicha")
plt.ylabel("Intención real")

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

### Interpretación de las matrices de confusión

La matriz de confusión global presenta una diagonal claramente predominante,
lo que indica que la mayor parte de las consultas fue asignada correctamente
a su intención. Este comportamiento es consistente con los 2,673 aciertos
obtenidos sobre las 3,080 consultas del conjunto de prueba y con la accuracy
global de 86.79 %.

No obstante, la matriz enfocada permite observar que los errores no se
distribuyen de manera uniforme entre las categorías. Algunas intenciones
presentan patrones de confusión claramente concentrados en categorías
semánticamente relacionadas.

El caso más destacado es `virtual_card_not_working`, donde únicamente 1 de
las 40 consultas fue clasificada correctamente. De sus 39 errores, 18 fueron
asignados a `getting_virtual_card` y 17 a `get_disposable_virtual_card`.
Esto muestra una dificultad específica para distinguir entre diferentes
intenciones relacionadas con tarjetas virtuales.

También se observa una confusión importante entre `why_verify_identity` y
`verify_my_identity`, con 16 consultas asignadas a esta última categoría.
Asimismo, `card_swallowed` presenta 15 predicciones como
`declined_cash_withdrawal`, mientras que `topping_up_by_card` distribuye
varios de sus errores entre otras categorías relacionadas con operaciones
de recarga.

En conjunto, las matrices respaldan el análisis realizado previamente: el
modelo presenta un buen desempeño global, pero mantiene dificultades
localizadas para separar algunas intenciones que comparten vocabulario,
entidades bancarias o significados próximos. Estos patrones resultan más
consistentes con la hipótesis de similitud semántica planteada durante el EDA
que con una explicación basada exclusivamente en el número de muestras de
entrenamiento.

## 2.11 Conclusiones de la clasificación con DistilRoBERTa

El fine-tuning de `distilroberta-base` permitió construir un clasificador para
las 77 intenciones de Banking77 con un desempeño global elevado. Sobre el
conjunto de prueba, formado por 3,080 consultas balanceadas entre las
categorías, el modelo obtuvo una accuracy de **86.79 %**, una precisión macro
de **88.38 %**, un recall macro de **86.79 %** y un F1-macro de **86.02 %**.

Los resultados de validación y prueba fueron similares. La mejor época de
entrenamiento alcanzó una accuracy de validación de 87.31 % y un F1-macro de
85.29 %. Además, durante las tres épocas disminuyeron tanto la pérdida de
entrenamiento como la pérdida de validación mientras aumentaban las métricas,
por lo que no se observaron señales evidentes de sobreajuste dentro del
intervalo evaluado.

El análisis por clase reveló, sin embargo, que el desempeño no es uniforme.
Algunas intenciones fueron clasificadas correctamente en las 40 consultas de
prueba, mientras que `virtual_card_not_working` presentó solamente un acierto
y 39 errores. Otras categorías, como `card_swallowed`, `card_acceptance`,
`why_verify_identity` y `topping_up_by_card`, también concentraron una cantidad
importante de errores.

La primera hipótesis planteada durante el EDA, relacionada con la cantidad de
muestras disponibles durante el entrenamiento, no mostró una asociación
monotónica estadísticamente significativa con el desempeño por clase. La
correlación de Spearman entre el número de muestras y el F1 fue de
**ρ = 0.1514**, con **p = 0.1887**.

En cambio, el análisis de errores y las matrices de confusión mostraron
patrones concentrados entre intenciones semánticamente próximas. Por ejemplo,
35 de los 39 errores de `virtual_card_not_working` fueron asignados a
`getting_virtual_card` o `get_disposable_virtual_card`, mientras que 16 de los
21 errores de `why_verify_identity` fueron clasificados como
`verify_my_identity`.

Estos resultados son coherentes con el análisis exploratorio, donde se observó
que numerosas intenciones comparten términos y conceptos bancarios. Aunque
DistilRoBERTa puede aprovechar el contexto completo de las consultas, algunas
categorías se diferencian mediante matices semánticos muy específicos que
continúan generando confusiones.

La siguiente sección utilizará `Falcon-7B-Instruct` para analizar una muestra
de estas predicciones incorrectas. El objetivo será estudiar si un modelo de
lenguaje generativo puede proporcionar explicaciones coherentes sobre los
elementos lingüísticos o semánticos que pudieron contribuir a las confusiones
observadas.

# 3. Análisis de errores mediante Falcon-7B-Instruct

Después de evaluar el clasificador basado en DistilRoBERTa, se identificaron
407 predicciones incorrectas sobre las 3,080 consultas del conjunto de prueba.
Aunque las métricas globales muestran un buen desempeño, el análisis por clase
evidenció errores asociados principalmente con intenciones semánticamente
cercanas.

En esta sección se utiliza **Falcon-7B-Instruct** como modelo de lenguaje para
analizar una muestra de estas predicciones incorrectas. El objetivo no es
corregir las predicciones de DistilRoBERTa, sino evaluar si un LLM puede generar
explicaciones útiles sobre las posibles causas de confusión entre la intención
real y la intención predicha.

Falcon-7B-Instruct se carga mediante cuantización de 4 bits para reducir los
requerimientos de memoria y permitir su ejecución en la GPU disponible
(Technology Innovation Institute, n.d.; Hugging Face, n.d.).

El procedimiento comprende:

1. seleccionar 20 errores de clasificación;
2. cargar Falcon-7B-Instruct mediante cuantización de 4 bits;
3. diseñar y calibrar el prompt y los parámetros de generación;
4. generar explicaciones para los 20 errores seleccionados;
5. evaluar manualmente la calidad de las respuestas;
6. analizar las capacidades y limitaciones observadas en el LLM.

## 3.1 Selección de predicciones incorrectas

A partir de las predicciones erróneas identificadas en la sección anterior,
se seleccionan 20 consultas para el análisis con Falcon-7B-Instruct.

Para evitar que la muestra quede dominada por unas pocas categorías con muchos
errores, primero se selecciona una observación por cada intención real
(`true_intent`) y posteriormente se extraen 20 de estas observaciones. Se utiliza
la semilla definida previamente para que la selección sea reproducible.

Esta estrategia busca obtener una muestra diversa de errores y permite observar
el comportamiento del LLM frente a diferentes tipos de confusión entre clases.

In [ ]:
# Selección reproducible de 20 errores pertenecientes a distintas
# intenciones reales.

sample_20 = (
    errors_df
    .groupby("true_intent", group_keys=False)
    .sample(n=1, random_state=SEED)
    .sample(n=20, random_state=SEED)
    .reset_index(drop=True)
)

sample_20.insert(
    0,
    "sample_id",
    range(1, len(sample_20) + 1)
)

print(f"Errores seleccionados: {len(sample_20)}")
print(
    f"Intenciones reales diferentes: "
    f"{sample_20['true_intent'].nunique()}"
)

display(
    sample_20[
        [
            "sample_id",
            "text",
            "true_intent",
            "predicted_intent"
        ]
    ]
)

La muestra contiene 20 consultas correspondientes a 20 intenciones reales
diferentes. Por tanto, el análisis posterior no se concentra únicamente en las
clases con mayor número de errores, sino que incluye distintos tipos de
confusión observados en el clasificador.

Para cada consulta se conservarán tanto la intención real como la predicción
incorrecta de DistilRoBERTa. Estos dos valores, junto con el texto original,
constituirán el contexto proporcionado posteriormente a Falcon-7B-Instruct.

## 3.2 Preparación y carga de Falcon-7B-Instruct

Falcon-7B-Instruct posee aproximadamente 7 mil millones de parámetros, por lo
que su carga directa en precisión completa supera la memoria disponible en una
GPU Tesla T4.

Para hacer viable el experimento en Google Colab se utilizará cuantización de
4 bits mediante `bitsandbytes`. Esta técnica reduce considerablemente el uso de
VRAM sin modificar los pesos mediante un nuevo proceso de entrenamiento.

Antes de cargar Falcon se eliminan de la memoria GPU los objetos asociados con
DistilRoBERTa que ya no son necesarios para las siguientes etapas.

In [ ]:
# Liberación de modelos que puedan permanecer cargados en GPU.
# A partir de este punto solo se conservan los resultados necesarios
# para el análisis posterior.

for variable_name in [
    "model",
    "trainer",
    "falcon_model"
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()

free_memory, total_memory = torch.cuda.mem_get_info()

print(
    f"VRAM disponible: "
    f"{free_memory / 1024**3:.2f} GB / "
    f"{total_memory / 1024**3:.2f} GB"
)

### 3.2.1 Carga cuantizada del modelo

Se utiliza `tiiuae/falcon-7b-instruct`, un modelo de lenguaje de aproximadamente
7 mil millones de parámetros. Debido a las restricciones de memoria de la GPU
Tesla T4, el modelo se carga mediante cuantización de 4 bits con
`bitsandbytes`.

Se emplea cuantización NF4 con doble cuantización y cálculo en `float16`.
Esta configuración permite reducir el consumo de memoria y ejecutar el modelo
completo en la GPU disponible, sin realizar un nuevo entrenamiento de sus
parámetros.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

FALCON_MODEL_NAME = "tiiuae/falcon-7b-instruct"

# Configuración de cuantización para reducir el consumo de VRAM.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Tokenizador de Falcon.
falcon_tokenizer = AutoTokenizer.from_pretrained(
    FALCON_MODEL_NAME
)

# Falcon no define originalmente un token específico de padding.
if falcon_tokenizer.pad_token is None:
    falcon_tokenizer.pad_token = falcon_tokenizer.eos_token

# Carga cuantizada directamente sobre la GPU.
falcon_model = AutoModelForCausalLM.from_pretrained(
    FALCON_MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16
)

falcon_model.eval()

In [ ]:
# Verificación de la carga y del consumo de memoria.

free_memory, total_memory = torch.cuda.mem_get_info()

print(f"Modelo: {FALCON_MODEL_NAME}")
print(f"Dispositivo: {next(falcon_model.parameters()).device}")
print(
    f"Cuantización 4 bits: "
    f"{getattr(falcon_model, 'is_loaded_in_4bit', False)}"
)
print(
    f"VRAM utilizada: "
    f"{(total_memory - free_memory) / 1024**3:.2f} GB"
)
print(
    f"VRAM disponible: "
    f"{free_memory / 1024**3:.2f} GB"
)

## 3.3 Diseño inicial del prompt

El objetivo del prompt es solicitar a Falcon-7B-Instruct una explicación de
por qué una consulta clasificada incorrectamente por DistilRoBERTa podría haber
sido asociada con la intención predicha en lugar de la intención real.

Para ello se proporcionan tres elementos:

- la consulta original del usuario;
- la intención real (`true_intent`);
- la intención predicha incorrectamente (`predicted_intent`).

Como punto de partida se utiliza una instrucción breve y directa. Posteriormente
se evaluarán diferentes configuraciones de generación y refinamientos del prompt
antes de seleccionar la versión definitiva.

### 3.3.1 Prompt base

La primera versión proporciona el contexto de clasificación y solicita una
explicación breve. Se indica explícitamente que la respuesta debe basarse en la
consulta para reducir la incorporación de información ajena al texto original.

In [ ]:
def build_prompt_base(text, true_intent, predicted_intent):
    return f"""
You are analyzing an error made by a banking customer-support intent classifier.

Customer query:
"{text}"

Correct category: "{true_intent}"
Predicted category: "{predicted_intent}"

Explain why the customer's query could have been confused with the predicted
category instead of the correct category.

Use only information contained in the query.
Do not invent additional details.
Answer briefly.

Explanation:
""".strip()

In [ ]:
# Se utiliza la primera muestra como caso de referencia durante
# la calibración inicial.

calibration_sample = sample_20.iloc[0]

print("Consulta:")
print(calibration_sample["text"])

print("\nIntención real:")
print(calibration_sample["true_intent"])

print("\nPredicción de DistilRoBERTa:")
print(calibration_sample["predicted_intent"])

## 3.4 Calibración del prompt y parámetros de generación

Antes de generar las explicaciones de las 20 consultas seleccionadas, se realiza
una etapa de calibración. El propósito es observar cómo diferentes parámetros
de generación y modificaciones del prompt afectan la claridad, estabilidad y
utilidad de las respuestas producidas por Falcon-7B-Instruct.

Las pruebas se realizan inicialmente sobre una misma consulta de referencia para
que las diferencias observadas puedan atribuirse principalmente a la
configuración evaluada.

Se analizarán tres aspectos:

1. efecto de la temperatura sobre la variabilidad de la respuesta;
2. efecto de la longitud máxima de generación;
3. refinamiento progresivo de las instrucciones proporcionadas al modelo.

### 3.4.1 Efecto de la temperatura

La temperatura controla la distribución utilizada durante el muestreo de
tokens. Para observar su efecto se comparan tres valores (`0.2`, `0.5` y `0.8`)
manteniendo constantes el prompt, la consulta, `top_p` y la longitud máxima de
generación.

Se utiliza `do_sample=True`, ya que la temperatura solo afecta la generación
cuando se realiza muestreo. Además, se fija la misma semilla antes de cada
generación para favorecer una comparación reproducible.

Se espera que temperaturas bajas produzcan respuestas más conservadoras,
mientras que valores mayores pueden incrementar la variabilidad. El objetivo
no es asumir de antemano cuál es la mejor configuración, sino observar su
comportamiento en este problema.

In [ ]:
def generate_calibration(
    prompt,
    temperature=0.5,
    max_new_tokens=60,
    seed=SEED
):
    # Se reinicia la semilla en cada prueba para mejorar
    # la reproducibilidad de la comparación.
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    inputs = falcon_tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = falcon_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=falcon_tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return falcon_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [ ]:
base_prompt = build_prompt_base(
    text=calibration_sample["text"],
    true_intent=calibration_sample["true_intent"],
    predicted_intent=calibration_sample["predicted_intent"]
)

temperature_results = []

for temperature in [0.2, 0.5, 0.8]:

    explanation = generate_calibration(
        prompt=base_prompt,
        temperature=temperature,
        max_new_tokens=60
    )

    temperature_results.append({
        "temperature": temperature,
        "explanation": explanation
    })

temperature_results_df = pd.DataFrame(
    temperature_results
)

display(temperature_results_df)

**Interpretación.** Las tres configuraciones producen explicaciones
comprensibles y reconocen algún grado de relación entre la consulta y la
categoría predicha. Sin embargo, se observan diferencias en su comportamiento.

Con `temperature=0.2`, Falcon relaciona la expresión *"all of the money has
disappeared"* con `top_up_failed`, aunque posteriormente se desvía hacia una
explicación general del funcionamiento del clasificador. Con
`temperature=0.5`, la respuesta identifica de forma más clara que ambas
intenciones están relacionadas con una operación de *top up* y con un problema
durante la transacción. Con `temperature=0.8`, la explicación continúa siendo
coherente, pero introduce una interpretación adicional sobre lo que el cliente
pudo haber pensado que no está explícitamente sustentada por la consulta.

En los tres casos la respuesta alcanza el límite de generación antes de
completar adecuadamente la explicación. Por tanto, esta prueba no permite
atribuir el problema únicamente a la temperatura. Los resultados sugieren que
también es necesario evaluar la longitud máxima de generación y posteriormente
refinar la estructura del prompt.

Para la siguiente prueba se conserva `temperature=0.5` como valor intermedio,
no por considerarlo todavía óptimo, sino para aislar el efecto de la longitud
máxima de generación.

### 3.4.2 Efecto de la longitud máxima de generación

Las respuestas anteriores alcanzaron el límite de 60 nuevos tokens sin
completar la explicación. Para determinar si el truncamiento se debe
principalmente a esta restricción, se mantiene `temperature=0.5` y se incrementa
`max_new_tokens` a 100.

De esta manera se modifica una sola variable respecto a la configuración
intermedia del experimento anterior.

In [ ]:
length_results = []

for max_tokens in [60, 100]:

    explanation = generate_calibration(
        prompt=base_prompt,
        temperature=0.5,
        max_new_tokens=max_tokens
    )

    length_results.append({
        "max_new_tokens": max_tokens,
        "explanation": explanation
    })

length_results_df = pd.DataFrame(length_results)

display(length_results_df)

**Interpretación.** Al incrementar `max_new_tokens` de 60 a 100, Falcon puede
completar la respuesta que anteriormente quedaba truncada. Esto confirma que
parte del problema observado se debía al límite de longitud de generación.

Sin embargo, disponer de más tokens no mejora necesariamente la calidad de la
explicación. La respuesta completa introduce una interpretación sobre lo que el
cliente supuestamente considera más específico o relevante, información que no
se encuentra explícitamente en la consulta.

Por tanto, aumentar la longitud resuelve el truncamiento, pero no evita
inferencias no sustentadas ni garantiza una explicación más precisa. El
siguiente paso consiste en refinar las instrucciones del prompt para controlar
mejor el contenido y la estructura de la respuesta.

### 3.4.3 Refinamiento semántico y estructural del prompt

Los experimentos anteriores muestran que modificar únicamente los parámetros
de generación no es suficiente. Aunque una mayor longitud evita respuestas
truncadas, Falcon puede incorporar explicaciones que no están sustentadas por
la consulta.

Por esta razón se refina el prompt con tres restricciones adicionales:

- centrar la explicación en el significado de la situación descrita por el
  cliente y no únicamente en las palabras que forman los nombres de las clases;
- utilizar exclusivamente información presente en la consulta;
- limitar la respuesta a un máximo de dos oraciones.

El objetivo es obtener explicaciones más breves y semánticamente fundamentadas,
reduciendo inferencias sobre información que el usuario no proporcionó.

In [ ]:
def build_prompt_semantic(text, true_intent, predicted_intent):
    return f"""
You are analyzing an error made by a banking customer-support
intent classifier.

Customer query:
"{text}"

Correct category: "{true_intent}"
Predicted category: "{predicted_intent}"

Explain why the meaning of the query could be confused with the predicted
category rather than the correct category.

Focus on the customer's situation and meaning, not on matching individual
words with the category names.
Use only information stated in the query.
Do not invent additional details.
Answer in at most two short sentences.

Explanation:
""".strip()

In [ ]:
semantic_test_ids = [2, 9, 12]

semantic_results = []

for sample_id in semantic_test_ids:

    row = sample_20[
        sample_20["sample_id"] == sample_id
    ].iloc[0]

    prompt = build_prompt_semantic(
        text=row["text"],
        true_intent=row["true_intent"],
        predicted_intent=row["predicted_intent"]
    )

    explanation = generate_calibration(
        prompt=prompt,
        temperature=0.5,
        max_new_tokens=100
    )

    semantic_results.append({
        "sample_id": sample_id,
        "query": row["text"],
        "true_intent": row["true_intent"],
        "predicted_intent": row["predicted_intent"],
        "explanation": explanation
    })

semantic_results_df = pd.DataFrame(semantic_results)

display(semantic_results_df)

**Interpretación.** El refinamiento semántico mejora parcialmente el enfoque de
las respuestas, pero no garantiza explicaciones fundamentadas ni el cumplimiento
del formato solicitado.

En la muestra 2, Falcon identifica correctamente que la consulta describe un
problema con una tarjeta virtual desechable, pero no explica con claridad por
qué este significado pudo conducir específicamente a la categoría predicha.
En la muestra 9, el modelo introduce información no presente en la consulta,
como asumir que el tipo de cambio corresponde a un retiro de efectivo e incluso
proponer posibles errores del banco. En la muestra 12 reconoce la proximidad
semántica entre la llegada de una tarjeta y la estimación de su entrega, pero
produce un razonamiento redundante y parcialmente contradictorio.

Además, las respuestas exceden el límite solicitado de dos oraciones. Estos
resultados muestran que una instrucción explícita de no inventar información no
es suficiente para impedir inferencias no sustentadas y que la fluidez de una
respuesta no implica necesariamente que su explicación sea correcta.

En particular, algunos errores de clasificación pueden originarse en consultas
que no contienen información suficiente para distinguir con claridad entre dos
intenciones cercanas. Por ello, el siguiente refinamiento permitirá al modelo
reconocer explícitamente la falta de información en lugar de completar el
contexto mediante inferencias.

### 3.4.4 Manejo explícito de incertidumbre

Los resultados anteriores sugieren que Falcon tiende a completar información
ausente para construir una explicación. Sin embargo, algunas consultas pueden
ser genuinamente insuficientes para distinguir entre dos intenciones.

Se modifica el prompt para permitir explícitamente que el modelo indique que
una consulta es ambigua o está insuficientemente especificada cuando el texto
no proporciona evidencia para distinguir las categorías. El objetivo es
reducir la necesidad de inventar contexto ausente.

Esta modificación se evalúa sobre tres consultas con diferentes niveles de
información: las muestras 9, 15 y 18.

In [ ]:
def build_prompt_uncertainty(text, true_intent, predicted_intent):
    return f"""
You are analyzing an error made by a banking customer-support
intent classifier.

Customer query:
"{text}"

Correct category: "{true_intent}"
Predicted category: "{predicted_intent}"

Explain why the meaning of the customer query could have led the classifier
to predict the predicted category instead of the correct category.

Focus only on the customer's situation and meaning.
Use only information explicitly stated in the query.
Do not invent missing details.

If the query does not contain enough information to distinguish between
the two categories, explicitly state that the query is ambiguous or
underspecified instead of assuming missing context.

Do not decide which category is correct; both labels are already provided.
Answer in at most two short sentences.
Output only the explanation.

Explanation:
""".strip()

In [ ]:
uncertainty_test_ids = [9, 15, 18]

uncertainty_results = []

for sample_id in uncertainty_test_ids:

    row = sample_20[
        sample_20["sample_id"] == sample_id
    ].iloc[0]

    prompt = build_prompt_uncertainty(
        text=row["text"],
        true_intent=row["true_intent"],
        predicted_intent=row["predicted_intent"]
    )

    explanation = generate_calibration(
        prompt=prompt,
        temperature=0.5,
        max_new_tokens=100
    )

    uncertainty_results.append({
        "sample_id": sample_id,
        "query": row["text"],
        "true_intent": row["true_intent"],
        "predicted_intent": row["predicted_intent"],
        "explanation": explanation
    })

uncertainty_results_df = pd.DataFrame(
    uncertainty_results
)

display(uncertainty_results_df)

**Interpretación.** Permitir explícitamente que Falcon reconozca la falta de
información mejora el tratamiento de algunas consultas realmente
subespecificadas. Esto se observa especialmente en la muestra 18 (*"Why did I
get a fee?"*), donde el texto no permite determinar si la comisión procede de
un retiro de efectivo o de un pago con tarjeta.

Un comportamiento similar resulta razonable en la muestra 9, ya que la consulta
menciona únicamente un tipo de cambio sin identificar el tipo de operación.
Sin embargo, el mismo patrón aparece en la muestra 15, aunque esta contiene
señales semánticas concretas relacionadas con un menor, la necesidad de una
tarjeta y su incorporación a una cuenta.

Por tanto, la nueva instrucción reduce algunas inferencias no sustentadas, pero
introduce un efecto secundario: Falcon tiende a utilizar la ambigüedad como una
respuesta dominante. También persisten problemas de formato y algunas
afirmaciones inconsistentes sobre la predicción del clasificador.

Para evitar imponer una plantilla verbal basada en la ambigüedad, el último
refinamiento utilizará un ejemplo (*one-shot prompting*) que muestre al modelo
el tipo de razonamiento, nivel de detalle y extensión esperados.

### 3.4.5 Refinamiento final mediante *one-shot prompting*

Las pruebas anteriores muestran que las instrucciones explícitas no siempre son
suficientes para controlar la forma de la respuesta. En particular, indicar al
modelo que reconozca consultas ambiguas provocó una tendencia a repetir este
tipo de explicación incluso cuando existían señales semánticas adicionales.

Como último refinamiento se utiliza *one-shot prompting*: además de las
instrucciones, se proporciona un ejemplo de una explicación breve que distingue
entre dos intenciones utilizando únicamente la información disponible en la
consulta.

También se ajustan los parámetros de generación. Se utiliza una temperatura
baja (`0.1`) para reducir la variabilidad del muestreo,
`repetition_penalty=1.25` para penalizar secuencias repetitivas y un máximo de
70 nuevos tokens. Se incorporan además secuencias de parada para evitar que el
modelo continúe generando bloques adicionales.

La configuración se evalúa primero sobre las muestras 7, 9, 15 y 18. Estas
incluyen tanto consultas con información discriminativa como casos
genuinamente subespecificados.

In [ ]:
def build_prompt_final(text, true_intent, predicted_intent):
    return f"""
### Instruction:
Analyze why a banking intent classifier could have misclassified a user query.

Explain which meaning or missing information in the query could make the
predicted intent plausible instead of the true intent.

Use only information present in the query.
Do not invent missing context.
If the query does not provide enough evidence to distinguish the two intents,
you may state that briefly.

Provide a direct explanation in 1 or 2 sentences maximum.
Do not repeat standard templates.

### Example:
Query: "Why did I get a fee?"
True: "cash_withdrawal_charge"
Predicted: "card_payment_fee_charged"
Explanation: The query mentions a fee but does not specify whether it came from
an ATM cash withdrawal or a card payment, so the wording alone does not clearly
distinguish between the two intents.

### Task:
Query: "{text}"
True: "{true_intent}"
Predicted: "{predicted_intent}"
Explanation:
""".strip()

In [ ]:
def generate_explanation_final(prompt, seed=SEED):

    # La generación utiliza muestreo, por lo que se fija una semilla
    # para favorecer la reproducibilidad.
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    inputs = falcon_tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = falcon_model.generate(
            **inputs,
            max_new_tokens=70,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.25,
            do_sample=True,
            eos_token_id=falcon_tokenizer.eos_token_id,
            pad_token_id=falcon_tokenizer.pad_token_id,
            stop_strings=["\n\n", "###"],
            tokenizer=falcon_tokenizer
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return falcon_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [ ]:
final_test_ids = [7, 9, 15, 18]

final_prompt_test = []

for sample_id in final_test_ids:

    row = sample_20[
        sample_20["sample_id"] == sample_id
    ].iloc[0]

    prompt = build_prompt_final(
        text=row["text"],
        true_intent=row["true_intent"],
        predicted_intent=row["predicted_intent"]
    )

    explanation = generate_explanation_final(prompt)

    final_prompt_test.append({
        "sample_id": sample_id,
        "query": row["text"],
        "true_intent": row["true_intent"],
        "predicted_intent": row["predicted_intent"],
        "explanation": explanation
    })

final_prompt_test_df = pd.DataFrame(final_prompt_test)

display(final_prompt_test_df)

**Interpretación.** El uso de *one-shot prompting* junto con una longitud máxima
de 70 tokens, temperatura baja, penalización de repetición y secuencias de
parada mejora notablemente el control de la forma de las respuestas. Las cuatro
explicaciones son breves, completas y evitan los párrafos extensos observados
en experimentos anteriores.

Sin embargo, la mejora estructural no implica necesariamente una mejora
semántica. En la muestra 7, Falcon no aprovecha plenamente la expresión
*"card wouldn't remove"*, que aporta información relevante sobre una posible
retención de la tarjeta. En la muestra 9 interpreta la ambigüedad como una
diferencia entre una pregunta y una instrucción, cuando la distinción relevante
es el tipo de operación asociada al tipo de cambio. En la muestra 15 identifica
la presencia de un menor, pero no relaciona claramente esta señal con
`age_limit`. Finalmente, en la muestra 18 reconoce correctamente que falta
contexto, aunque nuevamente caracteriza la ambigüedad de una forma poco
relacionada con las categorías comparadas.

Estos resultados muestran una diferencia importante entre **cumplimiento de
formato** y **calidad de la explicación**. El modelo puede producir una
respuesta breve y lingüísticamente coherente sin identificar correctamente la
causa semántica de la confusión del clasificador.

Se adopta esta configuración para el experimento final porque proporciona
respuestas más controladas y comparables para la evaluación manual. No obstante,
su corrección semántica no se presupone y será evaluada explícitamente sobre las
20 muestras seleccionadas.

## 3.5 Configuración final seleccionada

A partir de los experimentos anteriores se selecciona como configuración final
el prompt *one-shot* definido en la sección 3.4.5 y los siguientes parámetros
de generación:

- `temperature = 0.1`
- `top_p = 0.9`
- `repetition_penalty = 1.25`
- `max_new_tokens = 70`
- `do_sample = True`
- secuencias de parada: `"\n\n"` y `"###"`
- semilla fija para favorecer la reproducibilidad.

La selección representa un compromiso entre brevedad, control estructural y
capacidad explicativa. Los experimentos mostraron que ninguna configuración
elimina completamente las inferencias incorrectas, por lo que las respuestas
del LLM no se considerarán explicaciones verdaderas del funcionamiento interno
de DistilRoBERTa, sino explicaciones lingüísticas plausibles que requieren
validación manual.

## 3.6 Generación de explicaciones para los 20 errores

Una vez seleccionada la configuración final, se aplica el mismo prompt y los
mismos parámetros de generación a las 20 predicciones incorrectas seleccionadas
en la sección 3.1.

Para cada caso se conserva la consulta original, la intención real, la intención
predicha por DistilRoBERTa y la explicación generada por Falcon-7B-Instruct.
No se realizan ajustes individuales del prompt para ninguna consulta, de manera
que todas las respuestas sean obtenidas bajo las mismas condiciones y puedan
ser evaluadas posteriormente con criterios comunes.

In [ ]:
falcon_results = []

for _, row in sample_20.iterrows():

    prompt = build_prompt_final(
        text=row["text"],
        true_intent=row["true_intent"],
        predicted_intent=row["predicted_intent"]
    )

    explanation = generate_explanation_final(prompt)

    falcon_results.append({
        "sample_id": row["sample_id"],
        "query": row["text"],
        "true_intent": row["true_intent"],
        "predicted_intent": row["predicted_intent"],
        "explanation": explanation
    })

falcon_results_df = pd.DataFrame(falcon_results)

display(falcon_results_df)

## 3.7 Evaluación manual de las explicaciones

Las explicaciones generadas por un LLM pueden ser lingüísticamente fluidas sin
que necesariamente sean correctas o estén fundamentadas en la consulta. Por
esta razón, las 20 respuestas se evalúan manualmente considerando conjuntamente
el texto original, la intención real, la intención predicha por DistilRoBERTa
y la explicación producida por Falcon.

Se utilizan cuatro criterios:

- **Coherencia (`coherent`)**: la explicación es comprensible y presenta un
  razonamiento lingüísticamente consistente.
- **Fundamentación (`grounded`)**: la explicación utiliza información presente
  en la consulta y no introduce contexto o hechos no proporcionados.
- **Explicación de la confusión (`explains_confusion`)**: la respuesta explica
  de forma pertinente por qué la consulta podría asociarse con la intención
  predicha en lugar de la intención real.
- **Cumplimiento de formato (`format_ok`)**: la respuesta está completa y
  respeta el límite solicitado de una o dos oraciones.

Los criterios se valoran manualmente como `Sí` o `No`. Esta evaluación no
pretende medir automáticamente la calidad lingüística del modelo, sino analizar
críticamente la utilidad de sus explicaciones para comprender los errores del
clasificador.

In [ ]:
manual_labels = pd.DataFrame({
    "sample_id": list(range(1, 21)),

    "coherent": [
        "Sí", "Sí", "Sí", "Sí", "Sí",
        "Sí", "Sí", "Sí", "Sí", "Sí",
        "Sí", "Sí", "Sí", "Sí", "Sí",
        "Sí", "Sí", "Sí", "Sí", "Sí"
    ],

    "grounded": [
        "Sí", "Sí", "Sí", "No", "Sí",
        "No", "Sí", "Sí", "No", "No",
        "Sí", "Sí", "No", "No", "Sí",
        "Sí", "No", "No", "Sí", "Sí"
    ],

    "explains_confusion": [
        "No", "No", "No", "No", "Sí",
        "No", "No", "No", "No", "Sí",
        "Sí", "Sí", "No", "No", "No",
        "Sí", "No", "No", "Sí", "No"
    ],

    "format_ok": [
        "Sí", "Sí", "Sí", "Sí", "Sí",
        "Sí", "Sí", "Sí", "Sí", "Sí",
        "Sí", "Sí", "Sí", "Sí", "Sí",
        "Sí", "No", "No", "Sí", "Sí"
    ]
})

falcon_evaluation_df = falcon_results_df.merge(
    manual_labels,
    on="sample_id",
    how="left"
)

display(falcon_evaluation_df)

In [ ]:
evaluation_summary = pd.DataFrame({
    "Criterio": [
        "Coherencia",
        "Fundamentación",
        "Explica la confusión",
        "Cumplimiento de formato"
    ],
    "Cumple": [
        (falcon_evaluation_df["coherent"] == "Sí").sum(),
        (falcon_evaluation_df["grounded"] == "Sí").sum(),
        (falcon_evaluation_df["explains_confusion"] == "Sí").sum(),
        (falcon_evaluation_df["format_ok"] == "Sí").sum()
    ]
})

evaluation_summary["Total"] = len(falcon_evaluation_df)

evaluation_summary["Porcentaje"] = (
    evaluation_summary["Cumple"]
    / evaluation_summary["Total"]
    * 100
).round(1)

display(evaluation_summary)

## 3.8 Análisis de los resultados de Falcon-7B-Instruct

La evaluación manual muestra una diferencia importante entre la calidad formal
de las respuestas y su utilidad como explicaciones de los errores del
clasificador.

El 100% de las respuestas fue considerado coherente, lo que indica que
Falcon-7B-Instruct genera texto comprensible y lingüísticamente consistente.
Asimismo, el 90% cumplió con el formato solicitado de una o dos oraciones.
Estos resultados muestran que la configuración final, basada en *one-shot
prompting*, temperatura baja, penalización de repetición y secuencias de parada,
permitió controlar adecuadamente la extensión y estructura de la generación.

Sin embargo, solamente el 60% de las explicaciones se consideró fundamentado
en la información disponible en la consulta. En el resto de los casos Falcon
introdujo interpretaciones o detalles que no estaban explícitamente presentes
en el texto original. Por ejemplo, ante una consulta relacionada únicamente
con el tipo de cambio, el modelo interpretó la ambigüedad como una diferencia
entre una pregunta y una instrucción, distinción que no corresponde a las
categorías analizadas.

La diferencia es todavía mayor al evaluar la capacidad de explicar el error de
clasificación. Solamente 6 de las 20 respuestas (30%) explicaron de manera
pertinente por qué la intención predicha podía resultar plausible frente a la
intención real. En numerosos casos Falcon recurrió a expresiones genéricas como
*"the query is too vague"*, *"the query is ambiguous"* o *"does not provide
enough context"*, sin identificar qué elementos semánticos relacionaban
específicamente las dos categorías.

No obstante, reconocer falta de información resultó útil en algunos casos.
Consultas como *"How long does a card delivery take?"* presentan una proximidad
semántica considerable entre `card_arrival` y `card_delivery_estimate`.
Asimismo, una consulta que solamente menciona una transacción no reconocida
puede carecer de información suficiente para distinguir entre
`direct_debit_payment_not_recognised` y `card_payment_not_recognised`. En estos
casos, señalar la ausencia de información discriminativa sí contribuye a
explicar la confusión.

En contraste, otras consultas contienen señales semánticas relevantes que
Falcon no aprovecha adecuadamente. Por ejemplo, en *"I was retrieving money and
my card wouldn't remove"*, la expresión *"wouldn't remove"* proporciona una
señal importante relacionada con `card_swallowed`; sin embargo, Falcon reduce
el caso a un problema genérico de tarjeta. Esto demuestra que reconocer una
consulta como ambigua no siempre constituye una explicación adecuada.

En conjunto, los resultados muestran que la fluidez lingüística y el
cumplimiento de instrucciones de formato no deben utilizarse como indicadores
directos de calidad explicativa. Falcon puede producir respuestas plausibles y
bien redactadas que, al ser contrastadas con la consulta y las categorías,
resultan incompletas o semánticamente poco pertinentes.

### 3.8.1 Implicaciones del uso del LLM para explicar errores

Las explicaciones generadas por Falcon-7B-Instruct deben interpretarse como
explicaciones lingüísticas *post hoc* y no como una descripción causal del
funcionamiento interno de DistilRoBERTa. Falcon no tiene acceso a los pesos,
logits, representaciones internas ni mecanismos de atención utilizados por el
clasificador durante la predicción. Únicamente recibe la consulta, la etiqueta
real y la etiqueta predicha.

Por tanto, una explicación convincente no demuestra que DistilRoBERTa haya
utilizado efectivamente los elementos mencionados por Falcon para producir su
predicción. El LLM puede proponer una interpretación plausible de la confusión,
pero también puede omitir señales relevantes o introducir relaciones que no
están sustentadas por la consulta.

Los resultados de la evaluación manual refuerzan esta limitación: aunque todas
las respuestas fueron coherentes y la mayoría cumplió el formato solicitado,
solo el 30% explicó satisfactoriamente la confusión observada. En consecuencia,
el uso de un LLM puede resultar útil como apoyo exploratorio para analizar
errores, pero sus explicaciones requieren validación humana antes de utilizarse
para extraer conclusiones sobre el comportamiento del clasificador.

## 3.9 Conclusiones del análisis con Falcon-7B-Instruct

El experimento con Falcon-7B-Instruct permitió evaluar el uso de un modelo de
lenguaje como herramienta de apoyo para interpretar errores producidos por un
clasificador de intenciones. La etapa de calibración mostró que la calidad de
las respuestas depende no solamente de parámetros de generación como la
temperatura o la longitud máxima, sino también de la estructura y precisión de
las instrucciones proporcionadas al modelo.

Las primeras configuraciones mostraron problemas de truncamiento, respuestas
extensas e inferencias no sustentadas. El refinamiento mediante *one-shot
prompting*, una temperatura baja, penalización de repetición y secuencias de
parada permitió obtener respuestas más breves y controladas. Esto se refleja en
la evaluación final, donde el 100% de las explicaciones fue considerado
coherente y el 90% cumplió con el formato solicitado.

Sin embargo, el control formal de la generación no garantizó la calidad
semántica de las explicaciones. Solamente el 60% de las respuestas permaneció
fundamentado en la información disponible en la consulta y únicamente el 30%
explicó satisfactoriamente por qué la intención predicha podía confundirse con
la intención real. Falcon recurrió con frecuencia a explicaciones genéricas
basadas en supuesta ambigüedad o falta de contexto, incluso en consultas que
contenían señales relevantes para diferenciar las categorías.

Al mismo tiempo, el experimento mostró que la ambigüedad sí constituye una
explicación válida en determinados casos. Algunas consultas carecen de
información suficiente para distinguir entre intenciones semánticamente muy
próximas. En estos casos, el LLM puede ayudar a identificar la información
discriminativa ausente y aportar una interpretación útil del error.

Por tanto, Falcon-7B-Instruct resultó útil como herramienta exploratoria para
formular hipótesis sobre los errores del clasificador, pero no como mecanismo
automático de explicación. Sus respuestas deben someterse a validación humana,
ya que una explicación lingüísticamente coherente puede ser incompleta,
irrelevante o incorporar información no sustentada por la consulta.

Finalmente, las explicaciones generadas deben considerarse interpretaciones
*post hoc*. Falcon no observa las representaciones internas ni el proceso de
decisión de DistilRoBERTa y, por consiguiente, no permite establecer las causas
internas que originaron una predicción determinada. Su principal utilidad en
este experimento consiste en complementar el análisis cuantitativo de errores
con una perspectiva lingüística que, utilizada críticamente, puede ayudar a
identificar relaciones semánticas y posibles áreas de mejora del clasificador.

# 4. Conclusiones generales

El desarrollo de esta actividad permitió analizar un problema de clasificación
de intenciones desde tres perspectivas complementarias: las características del
conjunto de datos, el desempeño de un modelo Transformer discriminativo y el
uso de un modelo de lenguaje generativo para analizar los errores de
clasificación.

El análisis exploratorio mostró que las consultas del conjunto Banking77 son,
en general, textos cortos y que numerosas categorías comparten vocabulario
relacionado con tarjetas, pagos, transferencias, retiros y recargas. Los
bigramas y trigramas permitieron observar que parte de la información
discriminativa aparece en combinaciones de palabras y no únicamente en términos
aislados. También se identificó un desbalance moderado entre las 77 clases,
aunque el conjunto de prueba contiene 40 observaciones por categoría. A partir
de estos resultados se plantearon dos posibles dificultades para el
clasificador: la diferente representación de las clases durante el
entrenamiento y la proximidad semántica entre algunas intenciones.

El ajuste fino de DistilRoBERTa produjo un desempeño global satisfactorio. En
el conjunto de prueba se obtuvo una accuracy de 86.79%, una precisión macro de
88.38%, un recall macro de 86.79% y un F1 macro de 86.02%. Además, los
resultados de validación y prueba fueron cercanos y, durante las tres épocas
evaluadas, la pérdida disminuyó mientras las métricas aumentaron, sin mostrar
evidencias claras de sobreajuste en ese intervalo.

Sin embargo, el análisis por clase mostró que las métricas globales ocultan
diferencias importantes. Algunas intenciones alcanzaron una clasificación
prácticamente perfecta, mientras que `virtual_card_not_working` obtuvo un
recall de apenas 2.5% y acumuló 39 errores de 40 consultas. También se
observaron dificultades importantes en categorías como `card_swallowed`,
`card_acceptance` y `why_verify_identity`.

El número de ejemplos disponibles durante el entrenamiento no resultó
suficiente para explicar de forma general estas diferencias. La correlación de
Spearman entre el número de observaciones de entrenamiento por clase y su F1
fue débil (rho = 0.1514) y no estadísticamente significativa (p = 0.1887).
Por tanto, aunque algunas de las clases con pocos ejemplos presentaron un
desempeño bajo, los resultados no proporcionan evidencia suficiente para
afirmar una relación monotónica general entre mayor cantidad de ejemplos y
mejor desempeño.

En cambio, el análisis de la matriz de confusión proporcionó evidencia
descriptiva de que la proximidad semántica entre intenciones constituye una
fuente importante de errores. Por ejemplo, `virtual_card_not_working` fue
confundida principalmente con `getting_virtual_card` y
`get_disposable_virtual_card`, mientras que `why_verify_identity` se confundió
con `verify_my_identity` y `unable_to_verify_identity`. Estos casos muestran que
el clasificador puede identificar correctamente el dominio general de la
consulta y, aun así, seleccionar una intención cercana pero incorrecta.

El análisis con Falcon-7B-Instruct permitió estudiar estos errores desde una
perspectiva lingüística. La calibración mostró que modificar la temperatura y
la longitud de generación no era suficiente para obtener explicaciones
confiables. El refinamiento progresivo del prompt y el uso de *one-shot
prompting* mejoraron principalmente el control de la estructura de las
respuestas.

La evaluación manual de las 20 explicaciones finales confirmó esta diferencia.
El 100% de las respuestas fue coherente y el 90% cumplió con el formato
solicitado, pero solamente el 60% permaneció fundamentado en la información
contenida en la consulta y únicamente el 30% explicó satisfactoriamente la
confusión entre la intención real y la predicha. En varios casos Falcon recurrió
a explicaciones genéricas basadas en ambigüedad o falta de contexto, mientras
que en otros omitió señales semánticas relevantes presentes en la consulta.
Por tanto, la fluidez de una respuesta generada no debe confundirse con su
validez explicativa.

En conjunto, los resultados sugieren que una posible línea de mejora para el
clasificador consiste en prestar especial atención a pares de intenciones
semánticamente próximas. En lugar de asumir que el problema se resolvería
únicamente aumentando de forma general el número de ejemplos, podrían
incorporarse ejemplos contrastivos dirigidos a las confusiones observadas,
especialmente consultas que destaquen las diferencias entre categorías como
`virtual_card_not_working`, `getting_virtual_card` y
`get_disposable_virtual_card`. También podría evaluarse posteriormente el uso
de técnicas de aumento de datos o estrategias específicas para las clases con
menor desempeño.

Finalmente, Falcon-7B-Instruct puede utilizarse como herramienta exploratoria
para proponer interpretaciones sobre los errores, pero sus respuestas requieren
validación humana. Las explicaciones generadas son interpretaciones lingüísticas
*post hoc* y no explicaciones causales del funcionamiento interno de
DistilRoBERTa. La combinación de análisis exploratorio, métricas por clase,
matrices de confusión y revisión crítica de explicaciones generadas por un LLM
proporcionó una visión más completa del problema que la obtenida mediante una
única métrica global de clasificación.

# 5. Referencias

Casanueva, I., Temčinas, T., Gerz, D., Henderson, M., & Vulić, I. (2020).
Efficient intent detection with dual sentence encoders. *Proceedings of the
2nd Workshop on Natural Language Processing for Conversational AI*, 38–45.
https://doi.org/10.18653/v1/2020.nlp4convai-1.5

Liu, Y., Ott, M., Goyal, N., Du, J., Joshi, M., Chen, D., Levy, O., Lewis, M.,
Zettlemoyer, L., & Stoyanov, V. (2019). *RoBERTa: A robustly optimized BERT
pretraining approach*. arXiv. https://doi.org/10.48550/arXiv.1907.11692

Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). *DistilBERT, a distilled
version of BERT: Smaller, faster, cheaper and lighter*. arXiv.
https://doi.org/10.48550/arXiv.1910.01108

Penedo, G., Malartic, Q., Hesslow, D., Cojocaru, R., Cappelli, A., Alobeidli,
H., Pannier, B., Almazrouei, E., & Launay, J. (2023). The RefinedWeb dataset
for Falcon LLM: Outperforming curated corpora with web data, and web data only.
*Advances in Neural Information Processing Systems, 36*.

Hugging Face. (n.d.). *DistilRoBERTa base* [Model card].
https://huggingface.co/distilbert/distilroberta-base

Technology Innovation Institute. (n.d.). *Falcon-7B-Instruct* [Model card].
Hugging Face. https://huggingface.co/tiiuae/falcon-7b-instruct

Hugging Face. (n.d.). *Bitsandbytes*. Hugging Face documentation.
https://huggingface.co/docs/bitsandbytes/main/